In [4]:
%pwd

'/content'

# Temporal Stability of Counterfactual Explanations Under Distributional Drift
## in a Safety-Critical Railway Cyber-Physical System

**Author:** Benjamin Gyimah Boateng — University of Maryland  
**Target:** Conference Paper (Paper 1 of 2)  

---

### Notebook Structure
| Section | Description |
|---------|-------------|
| 1 | Setup & Data Loading |
| 2 | Feature Selection & Defect Labeling |
| 3 | Exploratory Data Analysis (EDA) |
| 4 | Distributional Drift Analysis |
| 5 | Information-Theoretic Analysis (MI + PID) |
| 6 | Baseline Model (XGBoost) |
| 7 | Minimal Counterfactual — Population Level |
| 8 | Temporal Transfer & Flip Rate Decay |
| 9 | Summary Tables & Paper-Ready Plots |

> **Scope of this notebook:** Paper 1 covers the *diagnosis* — demonstrating that counterfactual explanations degrade under distributional drift. The robust counterfactual formulation (Paper 2) is not implemented here.

---
## Section 1 — Setup & Data Loading

In [5]:
# ============================================================
# 1.1 — Core Imports
# ============================================================
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from scipy import stats

# Publication-quality plotting defaults
sns.set_theme(style='whitegrid', context='paper', font_scale=1.15)
sns.set_palette('colorblind')

plt.rcParams.update({
    # Resolution & layout
    'figure.dpi': 130,
    'savefig.dpi': 300,
    'figure.figsize': (8.5, 5),
    'savefig.bbox': 'tight',
    'savefig.pad_inches': 0.08,
    'savefig.facecolor': 'white',
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    # Fonts
    'font.family': 'sans-serif',
    'font.sans-serif': ['DejaVu Sans', 'Liberation Sans', 'Arial', 'Helvetica'],
    'mathtext.fontset': 'dejavusans',
    'font.size': 11,
    'axes.titlesize': 12,
    'axes.titleweight': 'bold',
    'axes.labelsize': 11,
    'axes.labelweight': 'regular',
    'xtick.labelsize': 9.5,
    'ytick.labelsize': 9.5,
    'legend.fontsize': 9.5,
    'legend.title_fontsize': 10,
    # Axes & spines
    'axes.linewidth': 1.0,
    'axes.edgecolor': '#333333',
    'axes.spines.top': False,
    'axes.spines.right': False,
    # Lines & markers
    'lines.linewidth': 1.8,
    'lines.markersize': 5,
    # Ticks
    'xtick.major.width': 1.0,
    'ytick.major.width': 1.0,
    'xtick.major.size': 4,
    'ytick.major.size': 4,
    'xtick.direction': 'out',
    'ytick.direction': 'out',
    # Grid
    'grid.color': '#cccccc',
    'grid.alpha': 0.55,
    'grid.linewidth': 0.5,
    'grid.linestyle': '--',
    # Legend
    'legend.frameon': True,
    'legend.framealpha': 0.95,
    'legend.edgecolor': '#cccccc',
    'legend.facecolor': 'white',
    # Image
    'image.cmap': 'viridis',
})

SEED = 42
np.random.seed(SEED)

print('Imports OK')


Imports OK


In [6]:
# ============================================================
# 1.2 — Load Raw Excel (local)
# ============================================================
import os

# Resolve project root relative to this notebook's working directory.
_cwd = os.getcwd()
if os.path.basename(_cwd) == 'notebooks':
    PROJECT_ROOT = os.path.dirname(_cwd)
else:
    PROJECT_ROOT = _cwd

PATH = os.path.join(PROJECT_ROOT, 'data', 'RawD.xlsx')
assert os.path.exists(PATH), f'Data file not found: {PATH}'

dfs_raw = pd.read_excel(PATH, sheet_name=None, engine='openpyxl')
print('Sheets found:', sorted(dfs_raw.keys()))


AssertionError: Data file not found: /content/data/RawD.xlsx

In [ ]:
# ============================================================
# 1.3 — Parse Monthly Sheets → Single Long DataFrame
# ============================================================
rx = re.compile(r'^(\d{1,2})_(\d{4})$')

def sheet_to_date(name: str) -> pd.Timestamp:
    m, y = map(int, rx.match(name).groups())
    return pd.Timestamp(year=y, month=m, day=1)

frames = []
for name, d in dfs_raw.items():
    if not rx.match(name):
        continue
    d = d.copy()
    d['date']  = sheet_to_date(name)
    d['year']  = d['date'].dt.year
    d['month'] = d['date'].dt.month
    frames.append(d)

raw_df = pd.concat(frames, ignore_index=True).sort_values('date').reset_index(drop=True)

# Clean integer columns
for c in ['MILE', 'FEET', 'TRACK']:
    if c in raw_df.columns:
        raw_df[c] = pd.to_numeric(raw_df[c], errors='coerce').astype('Int64')

print(f'Total rows   : {len(raw_df):,}')
print(f'Date range   : {raw_df["date"].min().date()} → {raw_df["date"].max().date()}')
print(f'Sheets loaded: {raw_df["date"].dt.to_period("M").nunique()} months')

---
## Section 2 — Feature Selection & Defect Labeling

Five geometry features are retained, matching the FRA Class 7 / proposed project limit thresholds used throughout the analysis.

| Feature | Physical Meaning | Proposed Limit |
|---------|-----------------|----------------|
| GAGE | Rail spacing (in) | 56.3 – 57.2 in |
| ALI_L_62 | Left 62-ft chord alignment (in) | ±0.20 in |
| ALI_R_62 | Right 62-ft chord alignment (in) | ±0.20 in |
| WARP_62 | 62-ft chord warp (in) | ±0.30 in |
| XLEVEL | Crosslevel deviation (in) | ±0.80 in |

A binary defect label is assigned: **1 = defective** if any feature exceeds its proposed limit.

In [ ]:
# ============================================================
# 2.1 — Select Features & Drop Missing
# ============================================================
FEATURES = ['GAGE', 'ALI_L_62', 'ALI_R_62', 'WARP_62', 'XLEVEL']
KEEP     = ['date', 'year', 'month', 'MILE', 'FEET'] + FEATURES

df = raw_df[KEEP].copy()
df = df.replace(-9999, np.nan)
df = df.dropna(subset=FEATURES).reset_index(drop=True)

print(f'Working rows : {len(df):,}')
print(f'Missing after drop: {df[FEATURES].isna().sum().sum()}')

In [ ]:
# ============================================================
# 2.2 — Defect Labeling (Proposed Project Limits)
# ============================================================
THRESHOLDS = {
    'GAGE'    : (56.3,  57.2),
    'ALI_L_62': (-0.20,  0.20),
    'ALI_R_62': (-0.20,  0.20),
    'WARP_62' : (-0.30,  0.30),
    'XLEVEL'  : (-0.80,  0.80),
}

df['defect'] = (
    (df['GAGE']     < THRESHOLDS['GAGE'][0])     |
    (df['GAGE']     > THRESHOLDS['GAGE'][1])     |
    (df['ALI_L_62'].abs() > THRESHOLDS['ALI_L_62'][1]) |
    (df['ALI_R_62'].abs() > THRESHOLDS['ALI_R_62'][1]) |
    (df['WARP_62'].abs()  > THRESHOLDS['WARP_62'][1])  |
    (df['XLEVEL'].abs()   > THRESHOLDS['XLEVEL'][1])
).astype(int)

# Summary
total   = len(df)
n_def   = df['defect'].sum()
print(f'Total samples  : {total:,}')
print(f'Defective (1)  : {n_def:,}  ({100*n_def/total:.2f}%)')
print(f'Non-defective  : {total-n_def:,}  ({100*(total-n_def)/total:.2f}%)')
print()
print('Defect rate by year:')
print(df.groupby('year')['defect'].agg(['sum','mean']).rename(columns={'sum':'n_defects','mean':'rate'}))

In [ ]:
# ============================================================
# 2.3 — Which Feature Drives Each Defect?
# ============================================================
defect_drivers = {}
for feat, (lo, hi) in THRESHOLDS.items():
    if feat == 'GAGE':
        mask = (df['GAGE'] < lo) | (df['GAGE'] > hi)
    else:
        mask = df[feat].abs() > hi
    defect_drivers[feat] = mask.sum()

driver_df = pd.Series(defect_drivers).sort_values(ascending=False)
print('Defects triggered by each feature (can overlap):')
print(driver_df)

fig, ax = plt.subplots(figsize=(7, 4))
driver_df.plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Defect Triggers by Feature')
ax.set_ylabel('Number of Triggered Defects')
ax.set_xlabel('')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

---
## Section 3 — Exploratory Data Analysis

This section characterizes the static properties of the dataset: feature distributions, the defect vs non-defective separation, spatial distribution of defects, and segment coverage across years.

In [ ]:
# ============================================================
# 3.1 — Descriptive Statistics
# ============================================================
print('=== Full Dataset Statistics ===')
display(df[FEATURES].describe(percentiles=[.90, .95, .99, .995]))

print('\n=== Skewness & Kurtosis ===')
sk_kurt = pd.DataFrame({
    'Skewness' : df[FEATURES].skew(),
    'Kurtosis' : df[FEATURES].kurt()
})
display(sk_kurt)

print('\n=== Absolute Value Stats (deviation features) ===')
display(df[['ALI_L_62','ALI_R_62','WARP_62','XLEVEL']].abs()
          .describe(percentiles=[.95, .99, .995]))

In [ ]:
# ============================================================
# 3.2 — Correlation Matrix
# ============================================================
corr = df[FEATURES].corr()

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(corr, annot=True, fmt='.3f', cmap='coolwarm',
            center=0, square=True, ax=ax,
            linewidths=0.5, cbar_kws={'shrink': 0.8})
ax.set_title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 3.3 — Feature Distributions: Defective vs Non-Defective
# ============================================================
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, feat in enumerate(FEATURES):
    ax = axes[i]
    lo, hi = THRESHOLDS[feat]
    sns.kdeplot(data=df, x=feat, hue='defect',
                fill=True, common_norm=False, alpha=0.45,
                palette={0: 'steelblue', 1: 'tomato'}, ax=ax)
    ax.axvline(lo, color='black', linestyle='--', linewidth=1, label='Threshold')
    ax.axvline(hi, color='black', linestyle='--', linewidth=1)
    ax.set_title(feat)
    ax.set_ylabel('Density')
    ax.legend(title='Defect', labels=['Non-Def', 'Defective'])

fig.delaxes(axes[-1])
fig.suptitle('Feature Distributions: Defective vs Non-Defective', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 3.4 — Spatial Coverage: Segment Consistency Across Years
# ============================================================
seg_years  = df.groupby(['MILE','FEET'])['year'].nunique()
total_yrs  = df['year'].nunique()
n_complete = (seg_years == total_yrs).sum()
n_total    = len(seg_years)

print(f'Total unique segments     : {n_total:,}')
print(f'Segments in all {total_yrs} years  : {n_complete:,}  ({100*n_complete/n_total:.1f}%)')
print(f'Mean observations/segment : {df.groupby(["MILE","FEET"]).size().mean():.1f}')

# Defect spatial scatter (all years combined)
defects_df = df[df['defect'] == 1]
fig, ax = plt.subplots(figsize=(10, 4))
sc = ax.scatter(defects_df['FEET'], defects_df['WARP_62'],
                c=defects_df['year'], cmap='plasma', alpha=0.5, s=10)
plt.colorbar(sc, ax=ax, label='Year')
ax.axhline(0.30,  color='red', linestyle='--', linewidth=1)
ax.axhline(-0.30, color='red', linestyle='--', linewidth=1)
ax.set_title('Spatial Distribution of Defects Along 1-Mile Segment (colored by year)')
ax.set_xlabel('FEET (position along mile)')
ax.set_ylabel('WARP_62')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 3.5 — Monthly Defect Rate Over Time
# ============================================================
monthly_defect = (
    df.groupby('date')['defect']
    .agg(rate='mean', count='sum')
    .reset_index()
)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(monthly_defect['date'], monthly_defect['rate'],
        marker='o', linewidth=2, color='tomato')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
plt.xticks(rotation=45)
ax.set_title('Monthly Defect Rate Over Time')
ax.set_xlabel('Date')
ax.set_ylabel('Defect Rate')
plt.tight_layout()
plt.show()

print('Yearly mean defect rate:')
print(df.groupby('year')['defect'].mean().round(4))

In [ ]:
# ============================================================
# 3.6 — Monthly TQI (Std Dev per Feature) — Track Quality Index
# ============================================================
tqi_cols = ['WARP_62', 'XLEVEL', 'GAGE', 'ALI_L_62', 'ALI_R_62']
tqi = df.groupby('date')[tqi_cols].std().reset_index()

fig, ax = plt.subplots(figsize=(11, 5))
markers = ['o', 's', '^', 'x', 'D']
for feat, mk in zip(tqi_cols, markers):
    ax.plot(tqi['date'], tqi[feat], marker=mk, linewidth=1.5,
            markersize=4, label=feat)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
plt.xticks(rotation=45)
ax.set_title('Monthly TQI (Std Dev) — 1 Mile Segment')
ax.set_xlabel('Date')
ax.set_ylabel('Standard Deviation')
ax.legend()
plt.tight_layout()
plt.show()

---
## Section 4 — Distributional Drift Analysis

This section quantifies how the geometry feature distributions evolve over time. Three complementary perspectives are examined:
1. **Variance drift** — does spread increase over time?
2. **Tail amplification** — does the probability of extreme values grow?
3. **Statistical tests** — are year-to-year distributions significantly different?

> These results directly motivate why a counterfactual computed in 2014 may fail to transfer to 2015 or 2016.

In [ ]:
# ============================================================
# 4.1 — Yearly Variance & 99th Percentile per Feature
# ============================================================
drift_stats = df.groupby('year')[FEATURES].agg(['std', lambda q: q.quantile(0.99)]).round(4)
drift_stats.columns = ['_'.join(c) for c in drift_stats.columns]
print('Yearly std and 99th percentile:')
display(drift_stats)

In [ ]:
# ============================================================
# 4.2 — Tail Exceedance Probability Over Time (WARP & XLEVEL)
# ============================================================
monthly_tail = df.groupby('date').apply(
    lambda g: pd.Series({
        'WARP_tail'  : (g['WARP_62'].abs() > 0.30).mean(),
        'XLEVEL_tail': (g['XLEVEL'].abs()   > 0.80).mean(),
    })
).reset_index()

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(monthly_tail['date'], monthly_tail['WARP_tail'],
        marker='o', linewidth=2, label='WARP Tail (|WARP|>0.30)')
ax.plot(monthly_tail['date'], monthly_tail['XLEVEL_tail'],
        marker='s', linewidth=2, label='XLEVEL Tail (|XLEVEL|>0.80)')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
plt.xticks(rotation=45)
ax.set_title('Monthly Tail Probability Over Time')
ax.set_xlabel('Date')
ax.set_ylabel('Tail Probability')
ax.legend()
plt.tight_layout()
plt.show()

print('Yearly tail exceedance rates:')
for yr in sorted(df['year'].unique()):
    w = (df[df['year']==yr]['WARP_62'].abs() > 0.30).mean()
    x = (df[df['year']==yr]['XLEVEL'].abs()   > 0.80).mean()
    print(f'  {yr}  |WARP|>0.30: {w:.4f}   |XLEVEL|>0.80: {x:.4f}')

In [ ]:
# ============================================================
# 4.3 — WARP Distribution Shift Across Years
# ============================================================
fig, ax = plt.subplots(figsize=(9, 5))
colors = ['#2196F3', '#FF9800', '#F44336', '#4CAF50']
for yr, col in zip(sorted(df['year'].unique()), colors):
    sns.kdeplot(data=df[df['year']==yr], x='WARP_62',
                fill=True, alpha=0.25, label=str(yr),
                color=col, ax=ax)
ax.axvline(-0.30, color='black', linestyle='--', linewidth=1)
ax.axvline( 0.30, color='black', linestyle='--', linewidth=1, label='±0.30 limit')
ax.set_title('WARP Distribution Drift by Year')
ax.set_xlabel('WARP_62')
ax.set_ylabel('Density')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 4.4 — Kolmogorov-Smirnov Tests: Year-to-Year Drift
#        (quantifies whether distributions are statistically different)
# ============================================================
from scipy.stats import ks_2samp

years  = sorted(df['year'].unique())
pairs  = [(years[i], years[i+1]) for i in range(len(years)-1)]

print('KS Test Results (WARP_62 and XLEVEL):')
print(f'{"Pair":<15} {"KS-stat WARP":>15} {"p-val WARP":>12} {"KS-stat XLEVEL":>16} {"p-val XLEVEL":>14}')
print('-'*75)
for y1, y2 in pairs:
    w1 = df[df['year']==y1]['WARP_62']
    w2 = df[df['year']==y2]['WARP_62']
    x1 = df[df['year']==y1]['XLEVEL']
    x2 = df[df['year']==y2]['XLEVEL']
    ks_w, p_w = ks_2samp(w1, w2)
    ks_x, p_x = ks_2samp(x1, x2)
    print(f'{y1}→{y2:<10} {ks_w:>15.4f} {p_w:>12.2e} {ks_x:>16.4f} {p_x:>14.2e}')

In [ ]:
# ============================================================
# 4.5 — 200-ft Window TQI Heatmap
#        Shows which spatial windows degrade fastest
# ============================================================
df['window_200ft'] = (df['FEET'] // 200).astype(int)

tqi_win = (
    df.groupby(['date', 'window_200ft'])['WARP_62']
    .std()
    .reset_index()
    .pivot(index='window_200ft', columns='date', values='WARP_62')
)

fig, ax = plt.subplots(figsize=(14, 5))
sns.heatmap(tqi_win, cmap='YlOrRd', ax=ax, cbar_kws={'label': 'Std Dev WARP'})
ax.set_title('200-ft Window WARP TQI Over Time')
ax.set_xlabel('Date')
ax.set_ylabel('200-ft Window ID')
# Simplify x-axis labels
n_cols = tqi_win.shape[1]
step   = max(1, n_cols // 10)
ax.set_xticks(range(0, n_cols, step))
ax.set_xticklabels(
    [str(tqi_win.columns[i].date()) for i in range(0, n_cols, step)],
    rotation=45, ha='right', fontsize=8
)
plt.tight_layout()
plt.show()

---
## Section 5 — Information-Theoretic Analysis (MI + PID)

Mutual Information (MI) quantifies how much each feature tells us about defect status.  
Partial Information Decomposition (PID) further decomposes the joint information of the two dominant features (WARP, XLEVEL) into:
- **Redundant** — shared information both carry
- **Unique WARP** — information only WARP carries
- **Unique XLEVEL** — information only XLEVEL carries
- **Synergy** — information only available when both are known together

> This analysis provides a principled basis for interpreting counterfactual directions: interventions should prioritize the structurally dominant feature.

In [ ]:
# ============================================================
# 5.1 — Mutual Information: Each Feature vs Defect
# ============================================================
from sklearn.feature_selection import mutual_info_classif

X_mi = df[FEATURES]
y_mi = df['defect']

mi_vals = mutual_info_classif(X_mi, y_mi, discrete_features=False, random_state=SEED)
mi_df   = pd.DataFrame({'Feature': FEATURES, 'MI': mi_vals})\
            .sort_values('MI', ascending=False)\
            .assign(Relative=lambda d: d['MI'] / d['MI'].max())

display(mi_df.round(4))

fig, ax = plt.subplots(figsize=(7, 4))
sns.barplot(data=mi_df, x='Feature', y='MI', palette='Blues_d', ax=ax)
ax.set_title('Mutual Information with Defect Status')
ax.set_ylabel('Mutual Information (nats)')
ax.set_xlabel('')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 5.2 — Partial Information Decomposition: WARP & XLEVEL → Defect
#        (Approximate PID via min-MI redundancy bound)
# ============================================================
mi_w     = mutual_info_classif(df[['WARP_62']], y_mi, random_state=SEED)[0]
mi_x     = mutual_info_classif(df[['XLEVEL']],  y_mi, random_state=SEED)[0]
mi_joint = mutual_info_classif(df[['WARP_62','XLEVEL']], y_mi, random_state=SEED)[0]

redundancy = min(mi_w, mi_x)
unique_w   = mi_w - redundancy
unique_x   = mi_x - redundancy
synergy    = mi_joint - mi_w - mi_x + redundancy

pid = {'Redundant': redundancy, 'Unique WARP': unique_w,
       'Unique XLEVEL': unique_x, 'Synergy': max(0, synergy)}

print('=== PID Decomposition (WARP, XLEVEL → Defect) ===')
for k, v in pid.items():
    print(f'  {k:<18}: {v:.4f}')

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].bar(pid.keys(), pid.values(), color='steelblue', edgecolor='white')
axes[0].set_title('PID Decomposition (absolute MI)')
axes[0].set_ylabel('Mutual Information')
plt.setp(axes[0].get_xticklabels(), rotation=20)

total_info = sum(pid.values())
pct = {k: 100*v/total_info for k,v in pid.items()}
axes[1].bar(pct.keys(), pct.values(), color='steelblue', edgecolor='white')
axes[1].set_title('PID Decomposition (%)')
axes[1].set_ylabel('% of Total Information')
plt.setp(axes[1].get_xticklabels(), rotation=20)

plt.suptitle('Partial Information Decomposition: WARP & XLEVEL → Defect', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 5.3 — Does WARP Dominance Hold Across Years?
#        (MI computed per year — checks structural stability)
# ============================================================
mi_by_year = {}
for yr in sorted(df['year'].unique()):
    sub = df[df['year'] == yr]
    mi  = mutual_info_classif(sub[FEATURES], sub['defect'],
                               discrete_features=False, random_state=SEED)
    mi_by_year[yr] = dict(zip(FEATURES, mi))

mi_yr_df = pd.DataFrame(mi_by_year).T
print('MI by year:')
display(mi_yr_df.round(4))

fig, ax = plt.subplots(figsize=(9, 4))
for feat in FEATURES:
    ax.plot(mi_yr_df.index, mi_yr_df[feat], marker='o', label=feat)
ax.set_title('Mutual Information by Year (Structural Stability Check)')
ax.set_xlabel('Year')
ax.set_ylabel('Mutual Information')
ax.legend()
plt.tight_layout()
plt.show()

---
## Section 6 — Baseline Model (XGBoost)

A single XGBoost classifier is trained on **2013–2014 data** and held fixed.  
The model is deliberately kept standard — the goal is to isolate the effect of distributional drift on counterfactual behavior, not to evaluate model performance under drift.

**Temporal split:**
- Training: 2013 + 2014
- Test Period 1 (mild drift): 2015
- Test Period 2 (severe drift): 2016

In [ ]:
# ============================================================
# 6.1 — Temporal Train/Test Split
# ============================================================
train_df    = df[df['year'].isin([2013, 2014])].copy()
test_2015   = df[df['year'] == 2015].copy()
test_2016   = df[df['year'] == 2016].copy()

X_train = train_df[FEATURES];  y_train = train_df['defect']
X_2015  = test_2015[FEATURES]; y_2015  = test_2015['defect']
X_2016  = test_2016[FEATURES]; y_2016  = test_2016['defect']

neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
SPW = neg / pos  # class imbalance weight

print(f'Train  : {len(train_df):>7,}  |  defect rate: {y_train.mean():.4f}')
print(f'2015   : {len(test_2015):>7,}  |  defect rate: {y_2015.mean():.4f}')
print(f'2016   : {len(test_2016):>7,}  |  defect rate: {y_2016.mean():.4f}')
print(f'scale_pos_weight: {SPW:.2f}')

In [ ]:
# ============================================================
# 6.2 — Train XGBoost (Fixed Reference Model)
# ============================================================
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, roc_auc_score, roc_curve, auc

model = XGBClassifier(
    n_estimators      = 300,
    max_depth         = 4,
    learning_rate     = 0.05,
    subsample         = 0.8,
    colsample_bytree  = 0.8,
    scale_pos_weight  = SPW,
    random_state      = SEED,
    eval_metric       = 'logloss',
    verbosity         = 0
)
model.fit(X_train, y_train)
print('Model trained.')

In [ ]:
# ============================================================
# 6.3 — Model Evaluation Across Time Periods
# ============================================================
splits = [
    ('Train (2013–2014)', X_train, y_train),
    ('Test 2015',         X_2015,  y_2015),
    ('Test 2016',         X_2016,  y_2016),
]

roc_results = {}
for label, X, y in splits:
    prob = model.predict_proba(X)[:,1]
    pred = model.predict(X)
    auc_val = roc_auc_score(y, prob)
    roc_results[label] = (roc_curve(y, prob), auc_val)
    print(f'=== {label} ===')
    print(classification_report(y, pred, digits=3))
    print(f'ROC AUC: {auc_val:.4f}\n')

# Combined ROC plot
fig, ax = plt.subplots(figsize=(7, 5))
for label, ((fpr, tpr, _), auc_val) in roc_results.items():
    ax.plot(fpr, tpr, linewidth=2, label=f'{label} (AUC={auc_val:.4f})')
ax.plot([0,1],[0,1],'k--', linewidth=1)
ax.set_title('ROC Curves — Temporal Evaluation')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

In [7]:
# ============================================================
# 6.4 — Feature Importance (XGBoost)
# ============================================================
imp_df = pd.DataFrame({'Feature': FEATURES,
                        'Importance': model.feature_importances_})\
           .sort_values('Importance', ascending=False)

fig, ax = plt.subplots(figsize=(7, 4))
sns.barplot(data=imp_df, x='Feature', y='Importance', palette='Blues_d', ax=ax)
ax.set_title('XGBoost Feature Importance')
ax.set_ylabel('Importance Score')
plt.tight_layout()
plt.show()
display(imp_df)

NameError: name 'FEATURES' is not defined

---
## Section 7 — Minimal Counterfactual (Population Level)

A counterfactual explanation is the **minimal perturbation** to a defective instance that changes the model prediction to non-defective:

$$\min_x \; \text{CE}(P_{\text{target}}) + \lambda \|x - x_0\|^2_w$$

Key design choices:
- **Reference regime**: CFs are computed on **2014 defective samples** only
- **Optimizer**: gradient-free coordinate search (compatible with XGBoost)
- **Cost weights**: GAGE penalized 5× (operationally harder to adjust)
- CFs are stored and applied unchanged to 2015 and 2016 in Section 8

In [8]:
# ============================================================
# 7.1 — CF Optimizer Setup
# ============================================================
BOUNDS = {
    'GAGE'    : (df['GAGE'].min(),     df['GAGE'].max()),
    'ALI_L_62': (df['ALI_L_62'].min(), df['ALI_L_62'].max()),
    'ALI_R_62': (df['ALI_R_62'].min(), df['ALI_R_62'].max()),
    'WARP_62' : (df['WARP_62'].min(),  df['WARP_62'].max()),
    'XLEVEL'  : (df['XLEVEL'].min(),   df['XLEVEL'].max()),
}

COST_W = np.array([5.0, 1.0, 1.0, 1.0, 1.0])  # GAGE penalized

def clip_bounds(x):
    x = x.copy()
    for i, f in enumerate(FEATURES):
        lo, hi = BOUNDS[f]
        x[i] = np.clip(x[i], lo, hi)
    return x

def weighted_l2(x, x0):
    return np.sum(COST_W * (x - x0)**2)

def ce_loss(p, eps=1e-9):
    p = np.clip(p, eps, 1 - eps)
    return -np.log(p)

def find_cf(model, x0, target=0, l2_weight=0.05,
             max_iter=2000, step_frac=0.10, restarts=30, seed=SEED):
    """Gradient-free coordinate search CF optimizer."""
    rng    = np.random.default_rng(seed)
    ranges = np.array([BOUNDS[f][1] - BOUNDS[f][0] for f in FEATURES])
    step0  = step_frac * ranges
    x0     = np.array(x0, dtype=float)

    best_x, best_obj = None, float('inf')

    for _ in range(restarts):
        x    = clip_bounds(x0 + rng.normal(0, 0.01, len(x0)) * ranges)
        step = step0.copy()

        for _ in range(max_iter):
            p_t  = model.predict_proba(x.reshape(1,-1))[0, 1-target] if target==0 \
                   else model.predict_proba(x.reshape(1,-1))[0, 1]
            obj  = ce_loss(p_t) + l2_weight * weighted_l2(x, x0)
            pred = int(model.predict(x.reshape(1,-1))[0])

            if pred == target and obj < best_obj:
                best_obj = obj
                best_x   = x.copy()

            improved = False
            for j in range(len(x)):
                for sgn in (-1, +1):
                    xt = clip_bounds(x.copy())
                    xt[j] += sgn * step[j]
                    xt = clip_bounds(xt)
                    p_try = model.predict_proba(xt.reshape(1,-1))[0, 1-target] if target==0 \
                            else model.predict_proba(xt.reshape(1,-1))[0, 1]
                    obj_t = ce_loss(p_try) + l2_weight * weighted_l2(xt, x0)
                    if obj_t < obj:
                        x, obj, improved = xt, obj_t, True
            if not improved:
                step *= 0.95
            if step.max() < 1e-6:
                break

    dist = np.sqrt(weighted_l2(best_x, x0)) if best_x is not None else None
    p_cf = model.predict_proba(best_x.reshape(1,-1))[0,1] if best_x is not None else None
    return best_x, dist, p_cf

print('CF optimizer ready.')

NameError: name 'df' is not defined

In [ ]:
# ============================================================
# 7.1 — CF Bounds: FRA Class 7 Hard Constraints
# ============================================================
# Three-tier constraint system:
#   PROJECT_LIMITS  → defect label definition (tightest)
#   FRA_LIMITS      → regulatory safety boundary (widest)
#   CF_BOUNDS       → search space = FRA ∩ observed data
#
# CFs are constrained to CF_BOUNDS — guaranteeing every
# suggested intervention is both legally compliant AND
# empirically plausible (within observed track conditions).
# ============================================================

PROJECT_LIMITS = {
    'GAGE'    : (56.3,  57.2),
    'ALI_L_62': (-0.20,  0.20),
    'ALI_R_62': (-0.20,  0.20),
    'WARP_62' : (-0.30,  0.30),
    'XLEVEL'  : (-0.80,  0.80),
}

FRA_LIMITS = {
    'GAGE'    : (56.0,  57.5),
    'ALI_L_62': (-0.50,  0.50),
    'ALI_R_62': (-0.50,  0.50),
    'WARP_62' : (-1.00,  1.00),
    'XLEVEL'  : (-1.00,  1.00),
}

# CF search bounds = most restrictive of FRA and observed data
CF_BOUNDS = {
    f: (
        max(FRA_LIMITS[f][0], df[f].min()),
        min(FRA_LIMITS[f][1], df[f].max())
    )
    for f in FEATURES
}

CF_BOUNDS_ARR  = np.array([[CF_BOUNDS[f][0], CF_BOUNDS[f][1]] for f in FEATURES])
CF_BOUNDS_LIST = [(CF_BOUNDS[f][0], CF_BOUNDS[f][1]) for f in FEATURES]

# MAD scaling for Gower-style distance (Wachter et al. formulation)
MAD = train_df[FEATURES].apply(
    lambda x: (x - x.median()).abs().median()
).values
MAD = np.where(MAD == 0, 1e-6, MAD)

# Cost weights: GAGE penalized (operationally constrained)
COST_W = np.array([5.0, 1.0, 1.0, 1.0, 1.0])

# Print bounds summary
print('=== CF Search Bounds ===')
print(f'{"Feature":<12} {"CF_LO":>8} {"CF_HI":>8} '
      f'{"FRA_LO":>8} {"FRA_HI":>8} '
      f'{"PROJ_LO":>9} {"PROJ_HI":>9} {"MAD":>8}')
print('-' * 72)
for i, f in enumerate(FEATURES):
    print(f'{f:<12} '
          f'{CF_BOUNDS[f][0]:>8.3f} {CF_BOUNDS[f][1]:>8.3f} '
          f'{FRA_LIMITS[f][0]:>8.3f} {FRA_LIMITS[f][1]:>8.3f} '
          f'{PROJECT_LIMITS[f][0]:>9.3f} {PROJECT_LIMITS[f][1]:>9.3f} '
          f'{MAD[i]:>8.4f}')

In [ ]:
# ============================================================
# DIAGNOSTIC — Why are CFs failing?
# ============================================================
import numpy as np

df_2014_def = df[(df['year'] == 2014) & (df['defect'] == 1)].copy()
X_def = df_2014_def[FEATURES].values

# 1. What does the model predict on raw defective samples?
probs = model.predict_proba(X_def)[:, 1]
print('=== Model confidence on 2014 defective samples ===')
print(f'Mean P(defect)  : {probs.mean():.3f}')
print(f'Median P(defect): {np.median(probs):.3f}')
print(f'Min P(defect)   : {probs.min():.3f}')
print(f'Max P(defect)   : {probs.max():.3f}')
print(f'P(defect) > 0.9 : {(probs > 0.9).sum()} samples')
print(f'P(defect) > 0.5 : {(probs > 0.5).sum()} samples')
print(f'P(defect) < 0.5 : {(probs < 0.5).sum()} samples  ← model already unsure')

# 2. Which features are violated per sample?
print('\n=== Feature violation breakdown ===')
for f in FEATURES:
    lo, hi = PROJECT_LIMITS[f]
    if f == 'GAGE':
        viol = ((df_2014_def[f] < lo) | (df_2014_def[f] > hi)).sum()
    else:
        viol = (df_2014_def[f].abs() > hi).sum()
    print(f'  {f:<12}: {viol:>4} violations ({100*viol/len(df_2014_def):.1f}%)')

# 3. How many samples have MULTIPLE feature violations?
viol_count = pd.Series(0, index=df_2014_def.index)
for f in FEATURES:
    lo, hi = PROJECT_LIMITS[f]
    if f == 'GAGE':
        viol_count += ((df_2014_def[f] < lo) | (df_2014_def[f] > hi)).astype(int)
    else:
        viol_count += (df_2014_def[f].abs() > hi).astype(int)

print('\n=== Number of simultaneous feature violations ===')
print(viol_count.value_counts().sort_index())

# 4. Test: what happens if we push ALL violated features to limit?
x_test = X_def.copy()
for j, f in enumerate(FEATURES):
    lo, hi = PROJECT_LIMITS[f]
    if f == 'GAGE':
        x_test[:, j] = np.where(x_test[:, j] < lo, lo + 0.001,
                       np.where(x_test[:, j] > hi, hi - 0.001, x_test[:, j]))
    else:
        x_test[:, j] = np.where(x_test[:, j] >  hi,  hi - 0.001,
                       np.where(x_test[:, j] < -hi, -hi + 0.001, x_test[:, j]))

pred_test = model.predict(x_test)
print(f'\n=== If ALL violated features pushed to project limit ===')
print(f'Still defective : {pred_test.sum()} / {len(pred_test)}')
print(f'Flipped to safe : {(pred_test == 0).sum()} / {len(pred_test)}')

# 5. What about pushing to FRA limits instead?
x_fra = X_def.copy()
for j, f in enumerate(FEATURES):
    lo, hi = FRA_LIMITS[f]
    if f == 'GAGE':
        x_fra[:, j] = np.where(x_fra[:, j] < lo, lo + 0.001,
                      np.where(x_fra[:, j] > hi, hi - 0.001, x_fra[:, j]))
    else:
        x_fra[:, j] = np.where(x_fra[:, j] >  hi,  hi - 0.001,
                      np.where(x_fra[:, j] < -hi, -hi + 0.001, x_fra[:, j]))

pred_fra = model.predict(x_fra)
print(f'\n=== If ALL violated features pushed to FRA limit ===')
print(f'Still defective : {pred_fra.sum()} / {len(pred_fra)}')
print(f'Flipped to safe : {(pred_fra == 0).sum()} / {len(pred_fra)}')

# 6. Check model decision boundary around WARP
print('\n=== Model sensitivity: WARP scan on median defective sample ===')
median_sample = np.median(X_def, axis=0)
warp_idx = FEATURES.index('WARP_62')
for warp_val in [-0.50, -0.40, -0.30, -0.20, -0.10, 0.0, 0.10, 0.20, 0.30]:
    x_scan = median_sample.copy()
    x_scan[warp_idx] = warp_val
    p = model.predict_proba(x_scan.reshape(1,-1))[0,1]
    pred = model.predict(x_scan.reshape(1,-1))[0]
    print(f'  WARP={warp_val:+.2f}  P(defect)={p:.4f}  pred={pred}')

In [ ]:
# ============================================================
# DIAGNOSTIC STEP 2 — Find True Model Decision Boundary
# for WARP and XLEVEL via binary search
# ============================================================

from scipy.optimize import brentq

def find_boundary(model, x_base, feat_idx, direction, lo, hi):
    """
    Binary search for exact WARP/XLEVEL value where
    model flips from defective to non-defective.
    direction: +1 for positive side, -1 for negative side
    """
    def pred_prob(val):
        x = x_base.copy()
        x[feat_idx] = val
        return model.predict_proba(x.reshape(1,-1))[0,1] - 0.5

    try:
        if direction == 1:
            boundary = brentq(pred_prob, 0.0, hi, xtol=1e-4)
        else:
            boundary = brentq(pred_prob, lo, 0.0, xtol=1e-4)
        return boundary
    except ValueError:
        return None

# Find boundary on median defective sample
X_def    = df_2014_def[FEATURES].values
x_median = np.median(X_def, axis=0)
warp_idx  = FEATURES.index('WARP_62')
xlev_idx  = FEATURES.index('XLEVEL')

warp_boundary_neg = find_boundary(model, x_median, warp_idx, -1,
                                   CF_BOUNDS['WARP_62'][0], 0.0)
warp_boundary_pos = find_boundary(model, x_median, warp_idx, +1,
                                   0.0, CF_BOUNDS['WARP_62'][1])
xlev_boundary_neg = find_boundary(model, x_median, xlev_idx, -1,
                                   CF_BOUNDS['XLEVEL'][0], 0.0)
xlev_boundary_pos = find_boundary(model, x_median, xlev_idx, +1,
                                   0.0, CF_BOUNDS['XLEVEL'][1])

print('=== True Model Decision Boundaries (median sample) ===')
print(f'WARP  negative side: {warp_boundary_neg:.4f}  '
      f'(project limit: -0.30, FRA limit: -1.00)')
print(f'WARP  positive side: {warp_boundary_pos:.4f}  '
      f'(project limit: +0.30, FRA limit: +1.00)')
print(f'XLEVEL negative side: {xlev_boundary_neg}  '
      f'(project limit: -0.80, FRA limit: -1.00)')
print(f'XLEVEL positive side: {xlev_boundary_pos}  '
      f'(project limit: +0.80, FRA limit: +1.00)')

In [ ]:
# ============================================================
# 7.2 — Per-Sample Boundary Search + CF Generation
# ============================================================
# Strategy:
# 1. For each defective sample, find the actual model
#    flip point for WARP via binary search (fast, ~20 calls)
# 2. Use that as the CF starting point
# 3. Nelder-Mead polish needs only ~50 iterations from there
# 4. For XLEVEL-only defects, use WARP adjustment as proxy
#    (justified by PID — XLEVEL has no unique information)
# ============================================================
from scipy.optimize import brentq, minimize
import time, warnings
warnings.filterwarnings('ignore')

def mad_distance(x, x0):
    return np.sum(np.abs(x - x0) / MAD)

def cf_objective(x, x0, model, l2_weight=0.5):
    x  = np.clip(x, CF_BOUNDS_ARR[:,0], CF_BOUNDS_ARR[:,1])
    p  = np.clip(model.predict_proba(x.reshape(1,-1))[0,1], 1e-9, 1-1e-9)
    ce = -np.log(1 - p)
    return ce + l2_weight * mad_distance(x, x0)

def find_warp_flip_point(model, x0, warp_idx):
    """
    Binary search for exact WARP value that flips prediction.
    Only ~20 model calls. Returns flip point or None.
    """
    warp_val = x0[warp_idx]

    def pred_prob(val):
        x = x0.copy()
        x[warp_idx] = val
        return model.predict_proba(x.reshape(1,-1))[0,1] - 0.5

    try:
        if warp_val < 0:
            # Search from current value toward 0
            lo = max(CF_BOUNDS['WARP_62'][0], warp_val)
            hi = 0.0
            # Check if boundary exists in this range
            if pred_prob(lo) * pred_prob(hi) < 0:
                return brentq(pred_prob, lo, hi, xtol=1e-4)
            # Try positive side (unlikely but check)
            if pred_prob(0.0) * pred_prob(CF_BOUNDS['WARP_62'][1]) < 0:
                return brentq(pred_prob, 0.0, CF_BOUNDS['WARP_62'][1], xtol=1e-4)
        else:
            lo = 0.0
            hi = min(CF_BOUNDS['WARP_62'][1], warp_val)
            if pred_prob(lo) * pred_prob(hi) < 0:
                return brentq(pred_prob, lo, hi, xtol=1e-4)
            if pred_prob(CF_BOUNDS['WARP_62'][0]) * pred_prob(0.0) < 0:
                return brentq(pred_prob, CF_BOUNDS['WARP_62'][0], 0.0, xtol=1e-4)
    except ValueError:
        pass
    return None

def find_cf(model, x0, l2_weight=0.5, seed=SEED):
    """
    Two-stage CF finder using true model boundary as init.
    Stage 1: Binary search → find exact WARP flip point (~20 calls)
    Stage 2: Nelder-Mead polish from flip point (~50 calls)
    Total: ~70-100 calls vs 600,000 in original code.
    """
    rng      = np.random.default_rng(seed)
    x0       = np.array(x0, dtype=float)
    warp_idx = FEATURES.index('WARP_62')
    candidates = []

    # --- Stage 1: Binary search for WARP flip point ---
    flip_val = find_warp_flip_point(model, x0, warp_idx)

    if flip_val is not None:
        # Build init: original values, WARP set to just past flip point
        x_init = x0.copy()
        # Push 10% further past the boundary to be safely non-defective
        margin = 0.05 * abs(flip_val - x0[warp_idx])
        x_init[warp_idx] = flip_val - np.sign(x0[warp_idx]) * margin
        x_init = np.clip(x_init, CF_BOUNDS_ARR[:,0], CF_BOUNDS_ARR[:,1])

        pred_init = int(model.predict(x_init.reshape(1,-1))[0])
        if pred_init == 0:
            candidates.append((x_init.copy(),
                                cf_objective(x_init, x0, model, l2_weight)))

        # Stage 2: Nelder-Mead polish from this informed start
        for noise_scale in [0.0, 0.005, 0.01]:
            x_start = x_init + rng.normal(0, noise_scale, len(x0)) * \
                      (CF_BOUNDS_ARR[:,1] - CF_BOUNDS_ARR[:,0])
            x_start = np.clip(x_start, CF_BOUNDS_ARR[:,0], CF_BOUNDS_ARR[:,1])

            res   = minimize(cf_objective, x_start,
                             args=(x0, model, l2_weight),
                             method='Nelder-Mead',
                             options={'maxiter': 100, 'xatol':1e-5, 'fatol':1e-5})
            x_res = np.clip(res.x, CF_BOUNDS_ARR[:,0], CF_BOUNDS_ARR[:,1])
            pred  = int(model.predict(x_res.reshape(1,-1))[0])
            if pred == 0:
                candidates.append((x_res.copy(), res.fun))

    # --- Fallback: wider Nelder-Mead search if binary search failed ---
    # Handles XLEVEL-only defects and edge cases
    if not candidates:
        ranges = CF_BOUNDS_ARR[:,1] - CF_BOUNDS_ARR[:,0]
        for _ in range(5):
            x_rand = x0 + rng.normal(0, 0.02, len(x0)) * ranges
            x_rand = np.clip(x_rand, CF_BOUNDS_ARR[:,0], CF_BOUNDS_ARR[:,1])
            res    = minimize(cf_objective, x_rand,
                              args=(x0, model, l2_weight),
                              method='Nelder-Mead',
                              options={'maxiter': 200, 'xatol':1e-5, 'fatol':1e-5})
            x_res  = np.clip(res.x, CF_BOUNDS_ARR[:,0], CF_BOUNDS_ARR[:,1])
            pred   = int(model.predict(x_res.reshape(1,-1))[0])
            if pred == 0:
                candidates.append((x_res.copy(), res.fun))

    if not candidates:
        return None, None, None

    best_x = min(candidates, key=lambda c: c[1])[0]
    dist   = mad_distance(best_x, x0)
    p_cf   = model.predict_proba(best_x.reshape(1,-1))[0,1]
    return best_x, dist, p_cf


In [ ]:
# ============================================================
# 7.3 — Run on ALL 2014 Defective Samples
# ============================================================
df_2014_def = df[(df['year'] == 2014) & (df['defect'] == 1)].copy()
print(f'2014 defective samples: {len(df_2014_def):,}')
print('Running boundary-guided CF optimizer...\n')

cf_records, failed = [], 0
t0 = time.time()

for i, (idx, row) in enumerate(df_2014_def.iterrows()):
    x0 = row[FEATURES].astype(float).values
    cf_x, cf_dist, cf_p = find_cf(model, x0)

    if cf_x is None:
        failed += 1
        continue

    delta = cf_x - x0
    cf_records.append({
        'orig_idx'  : idx,
        'MILE'      : row['MILE'],
        'FEET'      : row['FEET'],
        'date_orig' : row['date'],
        **{f'x0_{f}': x0[j]    for j, f in enumerate(FEATURES)},
        **{f'cf_{f}': cf_x[j]  for j, f in enumerate(FEATURES)},
        **{f'd_{f}' : delta[j] for j, f in enumerate(FEATURES)},
        'cf_dist'   : cf_dist,
        'cf_p_def'  : cf_p,
    })

    if (i + 1) % 100 == 0:
        elapsed   = time.time() - t0
        remaining = (len(df_2014_def) - i - 1) / ((i + 1) / elapsed)
        print(f'  {i+1}/{len(df_2014_def)} | '
              f'{elapsed:.0f}s elapsed | ~{remaining:.0f}s remaining')

cf_df = pd.DataFrame(cf_records)
print(f'\nDone in {time.time()-t0:.1f}s')
print(f'CFs found   : {len(cf_df):,}')
print(f'CFs failed  : {failed}')
print(f'Success rate: {100*len(cf_df)/(len(cf_df)+failed):.1f}%')

# Quick sanity check on delta directions
print('\n=== Mean |Δ| per feature ===')
for f in FEATURES:
    print(f'  {f:<12}: {cf_df[f"d_{f}"].abs().mean():.5f}')

In [ ]:
# ============================================================
# Who are the 12 failures?
# ============================================================
succeeded_idx = set(cf_df['orig_idx'].values)
all_idx       = set(df_2014_def.index.values)
failed_idx    = all_idx - succeeded_idx

failed_df = df_2014_def.loc[list(failed_idx), FEATURES + ['defect']].copy()

print('=== 12 Failed Samples ===')
print(failed_df[FEATURES].describe().round(4))

print('\nFeature violations in failed samples:')
for f in FEATURES:
    lo, hi = PROJECT_LIMITS[f]
    if f == 'GAGE':
        viol = ((failed_df[f] < lo) | (failed_df[f] > hi)).sum()
    else:
        viol = (failed_df[f].abs() > hi).sum()
    print(f'  {f:<12}: {viol} violations')

print('\nWARP values in failed samples:')
print(failed_df['WARP_62'].values.round(4))

print('\nXLEVEL values in failed samples:')
print(failed_df['XLEVEL'].values.round(4))

In [ ]:
# ============================================================
# 7.4 — FRA Compliance Verification (Hard Post-Filter)
# ============================================================
# Two-layer compliance check:
#   Layer 1: CF values within FRA Class 7 limits
#   Layer 2: CF values within observed data range
# Any CF failing either check is removed from cf_df.
# ============================================================

def is_fra_compliant(x_cf):
    """Returns True only if ALL features within FRA Class 7 limits."""
    for j, f in enumerate(FEATURES):
        lo, hi = FRA_LIMITS[f]
        if not (lo <= x_cf[j] <= hi):
            return False
    return True

def is_empirically_plausible(x_cf):
    """Returns True only if ALL features within observed data range."""
    for j, f in enumerate(FEATURES):
        lo = df[f].min()
        hi = df[f].max()
        if not (lo <= x_cf[j] <= hi):
            return False
    return True

# ---- Apply filters ----
n_before = len(cf_df)

fra_mask = cf_df.apply(
    lambda row: is_fra_compliant(
        np.array([row[f'cf_{f}'] for f in FEATURES])
    ), axis=1
)

plaus_mask = cf_df.apply(
    lambda row: is_empirically_plausible(
        np.array([row[f'cf_{f}'] for f in FEATURES])
    ), axis=1
)

combined_mask = fra_mask & plaus_mask
cf_df_clean   = cf_df[combined_mask].copy().reset_index(drop=True)
n_after       = len(cf_df_clean)
n_fra_fail    = (~fra_mask).sum()
n_plaus_fail  = (~plaus_mask).sum()
n_both_fail   = (~combined_mask).sum()

print('=== 7.4 — FRA Compliance & Plausibility Report ===\n')
print(f'CFs before filter          : {n_before:,}')
print(f'Failed FRA limits          : {n_fra_fail:,}')
print(f'Failed empirical range     : {n_plaus_fail:,}')
print(f'Failed either check        : {n_both_fail:,}')
print(f'CFs retained (compliant)   : {n_after:,}  '
      f'({100*n_after/n_before:.1f}%)')

# ---- Per-feature FRA violation detail ----
if n_fra_fail > 0:
    removed = cf_df[~fra_mask]
    print('\nFRA violations by feature:')
    for f in FEATURES:
        lo, hi = FRA_LIMITS[f]
        viol = ((removed[f'cf_{f}'] < lo) |
                (removed[f'cf_{f}'] > hi)).sum()
        if viol > 0:
            print(f'  {f:<12}: {viol} violations  '
                  f'(FRA limit: [{lo}, {hi}])')
else:
    print('\nNo FRA violations — all CFs compliant by construction.')

# ---- CF value summary ----
print('\n=== CF Value Ranges (compliant CFs only) ===')
print(f'{"Feature":<12} {"x0_mean":>9} {"cf_mean":>9} '
      f'{"delta_mean":>11} {"abs_delta":>10} '
      f'{"FRA_lo":>7} {"FRA_hi":>7} {"PROJ_hi":>8}')
print('-' * 78)
for f in FEATURES:
    fra_lo, fra_hi   = FRA_LIMITS[f]
    proj_lo, proj_hi = PROJECT_LIMITS[f]
    print(f'{f:<12} '
          f'{cf_df_clean[f"x0_{f}"].mean():>9.4f} '
          f'{cf_df_clean[f"cf_{f}"].mean():>9.4f} '
          f'{cf_df_clean[f"d_{f}"].mean():>11.5f} '
          f'{cf_df_clean[f"d_{f}"].abs().mean():>10.5f} '
          f'{fra_lo:>7.3f} {fra_hi:>7.3f} '
          f'{proj_hi:>8.3f}')

# ---- Distribution of CF WARP values vs limits ----
cf_warp = cf_df_clean['cf_WARP_62']
print(f'\n=== CF WARP Distribution ===')
print(f'Within project limit (±0.30) : '
      f'{(cf_warp.abs() <= 0.30).sum():,} / {n_after:,}  '
      f'({100*(cf_warp.abs() <= 0.30).mean():.1f}%)')
print(f'Between project & FRA limit  : '
      f'{((cf_warp.abs() > 0.30) & (cf_warp.abs() <= 1.00)).sum():,} / {n_after:,}  '
      f'({100*((cf_warp.abs() > 0.30) & (cf_warp.abs() <= 1.00)).mean():.1f}%)')
print(f'Outside FRA limit (>±1.00)   : '
      f'{(cf_warp.abs() > 1.00).sum():,} / {n_after:,}  '
      f'({100*(cf_warp.abs() > 1.00).mean():.1f}%)')

# ---- Visualization ----
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Panel 1: Distribution of CF WARP values
ax = axes[0]
ax.hist(cf_df_clean['x0_WARP_62'], bins=30, alpha=0.6,
        color='tomato', label='Original WARP')
ax.hist(cf_df_clean['cf_WARP_62'], bins=30, alpha=0.6,
        color='steelblue', label='CF WARP')
ax.axvline(-0.30, color='black', linestyle='--', linewidth=1,
           label='Project limit (±0.30)')
ax.axvline( 0.30, color='black', linestyle='--', linewidth=1)
ax.axvline(-1.00, color='red', linestyle=':', linewidth=1,
           label='FRA limit (±1.00)')
ax.axvline( 1.00, color='red', linestyle=':', linewidth=1)
ax.set_title('Original vs CF WARP Values')
ax.set_xlabel('WARP_62')
ax.set_ylabel('Count')
ax.legend(fontsize=8)

# Panel 2: Mean |Δ| per feature
ax = axes[1]
mean_abs_delta = pd.Series(
    {f: cf_df_clean[f'd_{f}'].abs().mean() for f in FEATURES}
).sort_values(ascending=False)
ax.bar(mean_abs_delta.index, mean_abs_delta.values,
       color='steelblue', edgecolor='white')
ax.set_title('Mean |Δ| per Feature\n(Minimal CF, 2014 Reference)')
ax.set_ylabel('Mean Absolute Change')
ax.set_xlabel('')
plt.setp(ax.get_xticklabels(), rotation=30)

plt.suptitle('CF Compliance & Feature Adjustment Summary',
             fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

# ---- Update cf_df to compliant version ----
cf_df = cf_df_clean.copy()
print(f'\ncf_df updated: {len(cf_df):,} FRA-compliant CFs ready for Section 8.')
print(f'12 XLEVEL-boundary failures excluded from transfer analysis.')

In [ ]:
# ============================================================
# SECTION 8 — Temporal Transfer & Flip Rate Decay
# ============================================================
# Core empirical contribution of Paper 1.
#
# Protocol:
#   1. Compute mean CF delta from 637 FRA-compliant 2014 CFs
#   2. Apply that fixed delta to ALL defective samples in
#      late 2014, 2015, and 2016
#   3. Measure flip rate per period — degradation = instability
#   4. Link decay mechanistically to WARP tail amplification
#
# Key distinction: the delta is FIXED from the reference
# regime (2014). It is NOT recomputed for later years.
# That is the entire point — we are testing whether a
# reference-regime explanation remains valid under drift.
# ============================================================

# ============================================================
# 8.1 — Compute Reference Delta from 2014 CFs
# ============================================================
delta_cols   = [f'd_{f}' for f in FEATURES]
mean_delta   = cf_df[delta_cols].mean().values
median_delta = cf_df[delta_cols].median().values

print('=== Reference Delta (mean of 637 FRA-compliant 2014 CFs) ===')
print(f'{"Feature":<12} {"Mean Δ":>10} {"Median Δ":>10} {"Mean |Δ|":>10}')
print('-' * 46)
for j, f in enumerate(FEATURES):
    print(f'{f:<12} '
          f'{mean_delta[j]:>+10.5f} '
          f'{median_delta[j]:>+10.5f} '
          f'{cf_df[f"d_{f}"].abs().mean():>10.5f}')

In [ ]:
# ============================================================
# 8.2 — Temporal Transfer: Flip Rate by Year
#        Two approaches compared:
#        A: Signed mean delta (exact 2014 reference)
#        B: Sign-corrected delta (magnitude preserved,
#           direction opposes violation)
# ============================================================

warp_idx = FEATURES.index('WARP_62')

def compute_flip_rate(model, year_df, delta, features, sign_correct=False):
    """
    Apply a fixed delta to all defective samples.
    If sign_correct=True, flip WARP delta sign based on
    violation direction of each sample.
    Returns flip rate, n_flipped, n_defective.
    """
    defective = year_df[year_df['defect'] == 1].copy()
    if len(defective) == 0:
        return 0.0, 0, 0

    X_orig = defective[features].values.astype(float)
    results = []

    for j in range(len(X_orig)):
        x     = X_orig[j].copy()
        d     = delta.copy()

        if sign_correct:
            # Flip WARP delta sign if sample violates positive side
            if x[warp_idx] > 0:
                d[warp_idx] = -abs(delta[warp_idx])
            else:
                d[warp_idx] = +abs(delta[warp_idx])

        x_cf = x + d
        # Enforce FRA bounds
        for k, f in enumerate(features):
            lo, hi = FRA_LIMITS[f]
            x_cf[k] = np.clip(x_cf[k], lo, hi)

        results.append(x_cf)

    X_cf      = np.array(results)
    pred_orig = model.predict(X_orig)
    pred_cf   = model.predict(X_cf)

    n_def     = (pred_orig == 1).sum()
    n_flipped = ((pred_orig == 1) & (pred_cf == 0)).sum()
    flip_rate = n_flipped / max(1, n_def)
    return flip_rate, int(n_flipped), int(n_def)


# ---- Define test periods ----
periods = [
    ('Late 2014', df[df['year'] == 2014]),
    ('2015',      df[df['year'] == 2015]),
    ('2016',      df[df['year'] == 2016]),
]

# ---- Run both approaches ----
results_A, results_B = [], []

for label, period_df in periods:
    # Option A: signed mean delta
    fr_A, n_flip_A, n_def_A = compute_flip_rate(
        model, period_df, mean_delta, FEATURES, sign_correct=False)

    # Option B: sign-corrected delta
    fr_B, n_flip_B, n_def_B = compute_flip_rate(
        model, period_df, mean_delta, FEATURES, sign_correct=True)

    results_A.append({
        'Period'       : label,
        'Defects'      : n_def_A,
        'Flipped'      : n_flip_A,
        'Not Flipped'  : n_def_A - n_flip_A,
        'Flip Rate (%)': round(100 * fr_A, 2),
    })
    results_B.append({
        'Period'       : label,
        'Defects'      : n_def_B,
        'Flipped'      : n_flip_B,
        'Not Flipped'  : n_def_B - n_flip_B,
        'Flip Rate (%)': round(100 * fr_B, 2),
    })

transfer_A = pd.DataFrame(results_A)
transfer_B = pd.DataFrame(results_B)

print('=== Option A: Signed Mean Delta (exact 2014 reference) ===')
display(transfer_A)

print('\n=== Option B: Sign-Corrected Delta (magnitude preserved) ===')
display(transfer_B)

# ---- Side by side flip rate comparison ----
print('\n=== Flip Rate Comparison ===')
print(f'{"Period":<12} {"Option A (%)":>14} {"Option B (%)":>14} {"Difference":>12}')
print('-' * 54)
for a, b in zip(results_A, results_B):
    diff = b['Flip Rate (%)'] - a['Flip Rate (%)']
    print(f'{a["Period"]:<12} '
          f'{a["Flip Rate (%)"]:>14.2f} '
          f'{b["Flip Rate (%)"]:>14.2f} '
          f'{diff:>+12.2f}')

In [ ]:
# ============================================================
# DIAGNOSTIC — Why are flip rates lower than expected?
# Examine what the delta actually does to 2015/2016 samples
# ============================================================

print('=== WARP violation direction breakdown by year ===')
for yr in [2014, 2015, 2016]:
    sub = df[(df['year'] == yr) & (df['defect'] == 1)]
    pos = (sub['WARP_62'] >  0.30).sum()
    neg = (sub['WARP_62'] < -0.30).sum()
    xlev = (sub['XLEVEL'].abs() > 0.80).sum()
    warp_ok = (sub['WARP_62'].abs() <= 0.30).sum()
    print(f'\n{yr}:')
    print(f'  Total defective     : {len(sub):,}')
    print(f'  WARP > +0.30        : {pos:,}  ({100*pos/len(sub):.1f}%)')
    print(f'  WARP < -0.30        : {neg:,}  ({100*neg/len(sub):.1f}%)')
    print(f'  WARP within ±0.30   : {warp_ok:,}  ({100*warp_ok/len(sub):.1f}%)')
    print(f'  XLEVEL violation    : {xlev:,}  ({100*xlev/len(sub):.1f}%)')

print('\n=== Mean delta magnitude vs mean violation magnitude ===')
print(f'Reference delta |WARP|  : {abs(mean_delta[warp_idx]):.5f}')
print(f'\nMean |WARP| violation by year (distance past ±0.30):')
for yr in [2014, 2015, 2016]:
    sub = df[(df['year'] == yr) & (df['defect'] == 1)]
    warp_def = sub[sub['WARP_62'].abs() > 0.30]['WARP_62']
    if len(warp_def) > 0:
        excess = (warp_def.abs() - 0.30).mean()
        mean_abs = warp_def.abs().mean()
        print(f'  {yr}: mean |WARP| = {mean_abs:.4f}  '
              f'mean excess past limit = {excess:.4f}  '
              f'n = {len(warp_def):,}')

print('\n=== Model boundary vs delta size ===')
print(f'True model flip point  : ~±0.283')
print(f'Project limit          : ±0.300')
print(f'Reference delta (WARP) : {mean_delta[warp_idx]:+.5f}')
print(f'If original WARP = -0.35, CF WARP = {-0.35 + mean_delta[warp_idx]:.4f}')
print(f'If original WARP = -0.40, CF WARP = {-0.40 + mean_delta[warp_idx]:.4f}')
print(f'If original WARP = -0.50, CF WARP = {-0.50 + mean_delta[warp_idx]:.4f}')
print(f'Model flips at         : ~-0.283')
print(f'\nConclusion: delta of {mean_delta[warp_idx]:+.5f} only sufficient'
      f' if original WARP > {-0.283 + mean_delta[warp_idx]:.4f}')

In [ ]:
# ============================================================
# SECTION 8 — Temporal Transfer & Flip Rate Decay
# Two transfer protocols compared:
#
# Protocol 1 — Mean Delta Transfer:
#   Apply the single mean delta from all 637 2014 CFs
#   to every defective sample in each test year.
#   Tests: does a population-level summary intervention
#   remain valid under drift?
#
# Protocol 2 — Nearest-Neighbor Matched Delta Transfer:
#   For each test-year defective sample, find the most
#   similar 2014 defective sample (by Euclidean distance)
#   and apply that sample's specific CF delta.
#   Tests: does an instance-matched reference CF remain
#   valid when applied to a similar future sample?
#
# Both protocols use ONLY 2014 reference information.
# The delta is never recomputed for test years.
# ============================================================

from sklearn.metrics import pairwise_distances

# Reference arrays from 2014 CFs
X_ref_orig  = cf_df[[f'x0_{f}' for f in FEATURES]].values  # original values
X_ref_delta = cf_df[[f'd_{f}'  for f in FEATURES]].values  # their deltas
X_ref_cf    = cf_df[[f'cf_{f}' for f in FEATURES]].values  # CF values

# Mean delta from Protocol 1
mean_delta = X_ref_delta.mean(axis=0)

print('Reference arrays ready.')
print(f'  X_ref_orig  : {X_ref_orig.shape}')
print(f'  X_ref_delta : {X_ref_delta.shape}')
print(f'  mean_delta  : {dict(zip(FEATURES, mean_delta.round(5)))}')

In [ ]:
# ============================================================
# 8.2 — Transfer Functions
# ============================================================

def transfer_mean_delta(model, year_df, mean_delta,
                        features, sign_correct=True):
    """
    Protocol 1: Apply mean 2014 delta to all defective samples.
    sign_correct=True: flip WARP sign based on violation direction.
    """
    defective = year_df[year_df['defect'] == 1].copy()
    if len(defective) == 0:
        return 0.0, 0, 0, pd.DataFrame()

    X_orig = defective[features].values.astype(float)
    X_cf   = []

    for x in X_orig:
        d = mean_delta.copy()
        if sign_correct:
            if x[warp_idx] > 0:
                d[warp_idx] = -abs(mean_delta[warp_idx])
            else:
                d[warp_idx] = +abs(mean_delta[warp_idx])
        x_cf = np.clip(x + d,
                       [FRA_LIMITS[f][0] for f in features],
                       [FRA_LIMITS[f][1] for f in features])
        X_cf.append(x_cf)

    X_cf      = np.array(X_cf)
    pred_orig = model.predict(X_orig)
    pred_cf   = model.predict(X_cf)

    flipped   = (pred_orig == 1) & (pred_cf == 0)
    n_def     = (pred_orig == 1).sum()
    n_flipped = flipped.sum()

    detail_df = defective.copy()
    detail_df['flipped'] = flipped
    detail_df['p_cf']    = model.predict_proba(X_cf)[:, 1]

    return n_flipped/max(1,n_def), int(n_flipped), int(n_def), detail_df


def transfer_nn_delta(model, year_df, X_ref_orig,
                      X_ref_delta, features, sign_correct=True):
    """
    Protocol 2: For each defective sample, find most similar
    2014 reference sample and apply its specific delta.
    Similarity measured by MAD-scaled Euclidean distance.
    """
    defective = year_df[year_df['defect'] == 1].copy()
    if len(defective) == 0:
        return 0.0, 0, 0, pd.DataFrame()

    X_orig = defective[features].values.astype(float)

    # MAD-scaled distance for fair comparison across features
    X_orig_scaled = X_orig / MAD
    X_ref_scaled  = X_ref_orig / MAD

    # Find nearest 2014 reference sample for each test sample
    dists   = pairwise_distances(X_orig_scaled, X_ref_scaled,
                                  metric='euclidean')
    nn_idx  = np.argmin(dists, axis=1)
    nn_dist = dists[np.arange(len(X_orig)), nn_idx]

    X_cf = []
    for i, x in enumerate(X_orig):
        d = X_ref_delta[nn_idx[i]].copy()
        if sign_correct:
            if x[warp_idx] > 0:
                d[warp_idx] = -abs(d[warp_idx])
            else:
                d[warp_idx] = +abs(d[warp_idx])
        x_cf = np.clip(x + d,
                       [FRA_LIMITS[f][0] for f in features],
                       [FRA_LIMITS[f][1] for f in features])
        X_cf.append(x_cf)

    X_cf      = np.array(X_cf)
    pred_orig = model.predict(X_orig)
    pred_cf   = model.predict(X_cf)

    flipped   = (pred_orig == 1) & (pred_cf == 0)
    n_def     = (pred_orig == 1).sum()
    n_flipped = flipped.sum()

    detail_df = defective.copy()
    detail_df['flipped']  = flipped
    detail_df['nn_dist']  = nn_dist
    detail_df['nn_idx']   = nn_idx
    detail_df['p_cf']     = model.predict_proba(X_cf)[:, 1]

    return n_flipped/max(1,n_def), int(n_flipped), int(n_def), detail_df

print('Transfer functions ready.')

In [ ]:
# ============================================================
# 8.3 — Run Both Protocols Across All Periods
# ============================================================
periods = [
    ('Late 2014', df[df['year'] == 2014]),
    ('2015',      df[df['year'] == 2015]),
    ('2016',      df[df['year'] == 2016]),
]

results_P1, results_P2 = [], []
detail_P1,  detail_P2  = {}, {}

for label, period_df in periods:
    # Protocol 1 — mean delta
    fr1, n1, d1, det1 = transfer_mean_delta(
        model, period_df, mean_delta, FEATURES)
    results_P1.append({
        'Period'       : label,
        'Defects'      : d1,
        'Flipped'      : n1,
        'Not Flipped'  : d1 - n1,
        'Flip Rate (%)': round(100 * fr1, 2),
    })
    detail_P1[label] = det1

    # Protocol 2 — nearest neighbour
    fr2, n2, d2, det2 = transfer_nn_delta(
        model, period_df, X_ref_orig, X_ref_delta, FEATURES)
    results_P2.append({
        'Period'       : label,
        'Defects'      : d2,
        'Flipped'      : n2,
        'Not Flipped'  : d2 - n2,
        'Flip Rate (%)': round(100 * fr2, 2),
    })
    detail_P2[label] = det2

transfer_P1 = pd.DataFrame(results_P1)
transfer_P2 = pd.DataFrame(results_P2)

print('=== Protocol 1: Mean Delta Transfer ===')
display(transfer_P1)

print('\n=== Protocol 2: Nearest-Neighbour Matched Delta ===')
display(transfer_P2)

print('\n=== Flip Rate Comparison ===')
print(f'{"Period":<12} {"P1 Mean Δ (%)":>15} '
      f'{"P2 NN Δ (%)":>13} {"Difference":>12}')
print('-' * 55)
for r1, r2 in zip(results_P1, results_P2):
    diff = r2['Flip Rate (%)'] - r1['Flip Rate (%)']
    print(f'{r1["Period"]:<12} '
          f'{r1["Flip Rate (%)"]:>15.2f} '
          f'{r2["Flip Rate (%)"]:>13.2f} '
          f'{diff:>+12.2f}')

# ============================================================
# FIGURE 6 — Flip rate decay (P1 and P2)
# ============================================================
import os
os.makedirs('figs', exist_ok=True)

periods_label = [r['Period'] for r in results_P1]
p1_rates = [r['Flip Rate (%)'] for r in results_P1]
p2_rates = [r['Flip Rate (%)'] for r in results_P2]

fig, ax = plt.subplots(figsize=(7, 4.5))

# Reference regime shading
ax.axvspan(-0.25, 0.25, alpha=0.12, color='#1D9E75', zorder=0)
ax.text(0, 108, 'reference\nregime', ha='center',
        fontsize=8, color='#0F6E56', linespacing=1.4, zorder=5)

ax.plot(periods_label, p1_rates,
        marker='o', linewidth=2.5, markersize=9,
        color='#EF9F27', linestyle='--',
        label='P1: mean delta', zorder=3)
ax.plot(periods_label, p2_rates,
        marker='s', linewidth=2.5, markersize=9,
        color='#D4537E',
        label='P2: nearest-neighbour delta', zorder=3)

# P1 annotations — below markers
for i, val in enumerate(p1_rates):
    ax.annotate(f'{val:.1f}%',
                (periods_label[i], val),
                textcoords='offset points',
                xytext=(-26, -16),
                fontsize=9, color='#BA7517')

# P2 annotations — above markers, Late 2014 shifts right
offsets_p2 = [(10, 8), (-26, 8), (8, 8)]
for i, (val, (dx, dy)) in enumerate(zip(p2_rates, offsets_p2)):
    ax.annotate(f'{val:.1f}%',
                (periods_label[i], val),
                textcoords='offset points',
                xytext=(dx, dy),
                fontsize=9, color='#993556')

ax.set_ylim(0, 120)
ax.set_ylabel('Flip rate (%)', fontsize=11)
ax.set_xlabel('Time period', fontsize=11)
ax.legend(fontsize=9, framealpha=0.6, edgecolor='#D3D1C7',
          handlelength=2.2)
ax.grid(axis='y', linewidth=0.5, alpha=0.4)
sns.despine(ax=ax)

plt.tight_layout()
plt.savefig('figs/flip_rate.pdf', dpi=300, bbox_inches='tight')
plt.savefig('figs/flip_rate.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: figs/flip_rate.pdf  |  figs/flip_rate.png')

In [ ]:
# ============================================================
# DIAGNOSTIC — Why are flip rates lower than expected?
# Examine what the delta actually does to 2015/2016 samples
# ============================================================

print('=== WARP violation direction breakdown by year ===')
for yr in [2014, 2015, 2016]:
    sub = df[(df['year'] == yr) & (df['defect'] == 1)]
    pos = (sub['WARP_62'] >  0.30).sum()
    neg = (sub['WARP_62'] < -0.30).sum()
    xlev = (sub['XLEVEL'].abs() > 0.80).sum()
    warp_ok = (sub['WARP_62'].abs() <= 0.30).sum()
    print(f'\n{yr}:')
    print(f'  Total defective     : {len(sub):,}')
    print(f'  WARP > +0.30        : {pos:,}  ({100*pos/len(sub):.1f}%)')
    print(f'  WARP < -0.30        : {neg:,}  ({100*neg/len(sub):.1f}%)')
    print(f'  WARP within \u00B10.30   : {warp_ok:,}  ({100*warp_ok/len(sub):.1f}%)')
    print(f'  XLEVEL violation    : {xlev:,}  ({100*xlev/len(sub):.1f}%)')

print('\n=== Mean delta magnitude vs mean violation magnitude ===')
print(f'Reference delta |WARP|  : {abs(mean_delta[warp_idx]):.5f}')
print(f'\nMean |WARP| violation by year (distance past \u00B10.30):')
for yr in [2014, 2015, 2016]:
    sub = df[(df['year'] == yr) & (df['defect'] == 1)]
    warp_def = sub[sub['WARP_62'].abs() > 0.30]['WARP_62']
    if len(warp_def) > 0:
        excess = (warp_def.abs() - 0.30).mean()
        mean_abs = warp_def.abs().mean()
        print(f'  {yr}: mean |WARP| = {mean_abs:.4f}  '
              f'mean excess past limit = {excess:.4f}  '
              f'n = {len(warp_def):,}')

print('\n=== Model boundary vs delta size ===')
print(f'True model flip point  : ~\u00B10.283')
print(f'Project limit          : \u00B10.300')
print(f'Reference delta (WARP) : {mean_delta[warp_idx]:+.5f}')
print(f'If original WARP = -0.35, CF WARP = {-0.35 + mean_delta[warp_idx]:.4f}')
print(f'If original WARP = -0.40, CF WARP = {-0.40 + mean_delta[warp_idx]:.4f}')
print(f'If original WARP = -0.50, CF WARP = {-0.50 + mean_delta[warp_idx]:.4f}')
print(f'Model flips at         : ~-0.283')
print(f'\nConclusion: delta of {mean_delta[warp_idx]:+.5f} only sufficient'
      f' if original WARP > {-0.283 + mean_delta[warp_idx]:.4f}')

# ============================================================
# 8.6 \u2014 Pearson Correlation: Flip Rate vs Drift Metrics
# ============================================================

# Calculate monthly flip rate data and correlations (moved from cell 7e1322f0)
monthly_nn = []
for date, grp in df[df['year'].isin([2014,2015,2016])].groupby('date'):
    fr, n_flip, n_def, _ = transfer_nn_delta(
        model, grp, X_ref_orig, X_ref_delta, FEATURES)
    warp_tail   = (grp['WARP_62'].abs() > 0.30).mean()
    defect_rate = grp['defect'].mean()
    monthly_nn.append({
        'date'        : date,
        'flip_rate'   : fr,
        'n_def'       : n_def,
        'warp_tail'   : warp_tail,
        'defect_rate' : defect_rate,
    })

mf_df = pd.DataFrame(monthly_nn).sort_values('date')

# Correlations
corr_warp = mf_df['flip_rate'].corr(mf_df['warp_tail'])
corr_dr   = mf_df['flip_rate'].corr(mf_df['defect_rate'])
corr_ndef = mf_df['flip_rate'].corr(mf_df['n_def'])

print('\n=== Mechanistic Link: Flip Rate vs Drift Metrics ===\n')

print(f'Flip rate vs WARP tail probability : r = {corr_warp:.3f}')
print(f'Flip rate vs defect rate           : r = {corr_dr:.3f}')
print(f'Flip rate vs n_defective           : r = {corr_ndef:.3f}')

print('\nInterpretation:')
print('  Strong negative correlation between flip rate and')
print('  WARP tail probability confirms that CF validity')
print('  degrades mechanistically as tail drift amplifies.')


In [ ]:
# ============================================================
# FIGURE 7 \u2014 Monthly flip rate (P2-NN) vs WARP tail probability
# ============================================================
import os
os.makedirs('figs', exist_ok=True)

fig, ax1 = plt.subplots(figsize=(8, 4.5))

# \u2500\u2500 Primary axis: monthly flip rate \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500
ax1.plot(mf_df['date'], mf_df['flip_rate'] * 100,
         color='#D4537E', linewidth=2, marker='o',
         markersize=5, label='P2 flip rate (%)', zorder=3)
ax1.set_ylabel('Flip rate (%)', fontsize=11, color='#993556')
ax1.tick_params(axis='y', labelcolor='#993556')
ax1.set_ylim(0, 110)

# \u2500\u2500 Secondary axis: WARP tail probability \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500
ax2 = ax1.twinx()
ax2.fill_between(mf_df['date'], mf_df['warp_tail'] * 100,
                 alpha=0.18, color='#EF9F27', zorder=1)
ax2.plot(mf_df['date'], mf_df['warp_tail'] * 100,
         color='#EF9F27', linewidth=2, marker='s',
         markersize=5, linestyle='--',
         label='WARP tail prob (%)', zorder=2)
ax2.set_ylabel('WARP tail probability (%)', fontsize=11, color='#BA7517')
ax2.tick_params(axis='y', labelcolor='#BA7517')
ax2.set_ylim(0, 30)

# \u2500\u2500 Year boundary lines \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500
for yr in [2015, 2016]:
    ax1.axvline(pd.Timestamp(f'{yr}-01-01'),
                color='#B4B2A9', linewidth=0.8,
                linestyle=':', zorder=0)
    ax1.text(pd.Timestamp(f'{yr}-01-01'), 105, str(yr),
             ha='center', fontsize=8, color='#888780')

# \u2500\u2500 Correlation annotation \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500
ax1.text(0.97, 0.92, f'r = {corr_warp:.3f}',
         transform=ax1.transAxes,
         ha='right', va='top', fontsize=10,
         color='#444441',
         bbox=dict(boxstyle='round,pad=0.3',
                   facecolor='white', edgecolor='#D3D1C7',
                   alpha=0.85))

# \u2500\u2500 Axes formatting \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax1.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
plt.setp(ax1.get_xticklabels(), rotation=35, ha='right', fontsize=9)
ax1.set_xlabel('Date', fontsize=11)
ax1.grid(axis='y', linewidth=0.5, alpha=0.35)
sns.despine(ax=ax1, right=False)

# \u2500\u2500 Combined legend \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2,
           fontsize=9, framealpha=0.7,
           edgecolor='#D3D1C7', loc='upper right',
           bbox_to_anchor=(0.97, 0.82))

plt.tight_layout()
plt.savefig('figs/monthly_flip.pdf', dpi=300, bbox_inches='tight')
plt.savefig('figs/monthly_flip.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: figs/monthly_flip.pdf  |  figs/monthly_flip.png')

In [ ]:
# ============================================================
# 8.6 \u2014 Monthly Flip Rate + Mechanistic Correlation
# ============================================================
import matplotlib.dates as mdates

# The monthly_nn dataframe and correlations are now computed in cell '7e1322f0'
# This cell now only contains the plotting logic.

# Plot
fig, ax1 = plt.subplots(figsize=(12, 5))
ax1.plot(mf_df['date'], mf_df['flip_rate']*100,
         marker='o', color='steelblue',
         linewidth=2, label='Monthly Flip Rate % (P2-NN)')
ax1.set_ylabel('Flip Rate (%)', color='steelblue')
ax1.tick_params(axis='y', labelcolor='steelblue')
ax1.set_ylim(0, 100)

ax2 = ax1.twinx()
ax2.plot(mf_df['date'], mf_df['warp_tail'],
         marker='s', color='tomato',
         linewidth=2, linestyle='--',
         label='WARP Tail Probability')
ax2.set_ylabel('WARP Tail Probability', color='tomato')
ax2.tick_params(axis='y', labelcolor='tomato')

ax1.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax1.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
plt.xticks(rotation=45)
ax1.set_title(
    f'Monthly Flip Rate vs WARP Tail Probability\n'
    f'(Pearson r = {corr_warp:.3f})', fontsize=11)
ax1.set_xlabel('Date')

lines1, l1 = ax1.get_legend_handles_labels()
lines2, l2 = ax2.get_legend_handles_labels()
ax1.legend(lines1+lines2, l1+l2, loc='upper left')
plt.tight_layout()
plt.show()

### 8.4.1 — Monthly Flip Rate Calculation and Correlations (P2-NN)

This section computes the monthly flip rates for Protocol 2 (Nearest-Neighbor Matched Delta) and calculates their correlations with drift metrics like WARP tail probability and overall defect rate. These metrics are then used in subsequent analysis and plotting sections.

In [ ]:
# ============================================================
# SECTION 9 — Summary Tables & Paper-Ready Figures
# ============================================================
# Consolidates all results into publication-ready format.
# Four tables + one four-panel figure covering the full
# Paper 1 narrative:
#   Data → Drift → Information Structure → CF Degradation
# ============================================================

# ============================================================
# 9.1 — Table 1: Dataset Overview
# ============================================================
print('=' * 60)
print('TABLE 1: Dataset Overview by Year')
print('=' * 60)

t1 = df.groupby('year').agg(
    N            = ('defect', 'count'),
    N_defective  = ('defect', 'sum'),
    Defect_rate  = ('defect', 'mean'),
    WARP_std     = ('WARP_62', 'std'),
    WARP_99th    = ('WARP_62', lambda q: q.abs().quantile(0.99)),
    XLEVEL_std   = ('XLEVEL', 'std'),
    WARP_tail    = ('WARP_62', lambda x: (x.abs() > 0.30).mean()),
).round(4)

t1.columns = ['N', 'N Defective', 'Defect Rate',
              'WARP Std', 'WARP 99th |abs|',
              'XLEVEL Std', 'WARP Tail Prob']
display(t1)

# ============================================================
# 9.2 — Table 2: Information-Theoretic Structure
# ============================================================
print('\n' + '=' * 60)
print('TABLE 2: Mutual Information & PID Decomposition')
print('=' * 60)

t2_mi = mi_df[['Feature', 'MI', 'Relative']].copy()
t2_mi.columns = ['Feature', 'MI (nats)', 'Relative to WARP']
display(t2_mi.round(4))

print('\nPID Decomposition (WARP, XLEVEL → Defect):')
pid_table = pd.DataFrame({
    'Component'  : ['Redundant', 'Unique WARP',
                    'Unique XLEVEL', 'Synergy'],
    'MI (nats)'  : [pid['Redundant'], pid['Unique WARP'],
                    pid['Unique XLEVEL'], pid['Synergy']],
    '% of Total' : [
        100*pid['Redundant']/sum(pid.values()),
        100*pid['Unique WARP']/sum(pid.values()),
        100*pid['Unique XLEVEL']/sum(pid.values()),
        100*pid['Synergy']/sum(pid.values()),
    ]
}).round(4)
display(pid_table)

# ============================================================
# 9.3 — Table 3: CF Generation Summary (2014 Reference)
# ============================================================
print('\n' + '=' * 60)
print('TABLE 3: CF Generation Summary — 2014 Reference Regime')
print('=' * 60)

t3 = pd.DataFrame({
    'Metric': [
        'Total 2014 defective samples',
        'CFs successfully generated',
        'CFs failed (XLEVEL-only boundary)',
        'Success rate',
        'FRA-compliant CFs',
        'FRA compliance rate',
        'Mean CF distance (MAD-scaled L1)',
        'Mean |Δ| WARP',
        'Mean |Δ| XLEVEL',
        'Mean |Δ| GAGE',
        'Mean |Δ| ALI_L_62',
        'Mean |Δ| ALI_R_62',
    ],
    'Value': [
        649,
        637,
        12,
        '98.2%',
        637,
        '100.0%',
        f"{cf_df['cf_dist'].mean():.4f}",
        f"{cf_df['d_WARP_62'].abs().mean():.5f}",
        f"{cf_df['d_XLEVEL'].abs().mean():.5f}",
        f"{cf_df['d_GAGE'].abs().mean():.5f}",
        f"{cf_df['d_ALI_L_62'].abs().mean():.5f}",
        f"{cf_df['d_ALI_R_62'].abs().mean():.5f}",
    ]
})
display(t3)

# ============================================================
# 9.4 — Table 4: Temporal Transfer Results
# ============================================================
print('\n' + '=' * 60)
print('TABLE 4: Temporal CF Transferability')
print('=' * 60)

# Defect composition per year
comp_rows = []
for yr, label in [(2014,'Late 2014'),(2015,'2015'),(2016,'2016')]:
    sub = df[(df['year']==yr) & (df['defect']==1)]
    n   = len(sub)
    comp_rows.append({
        'Period'          : label,
        'N Defective'     : n,
        'WARP<-0.30 (%)'  : round(100*(sub['WARP_62']<-0.30).sum()/n,1),
        'WARP>+0.30 (%)'  : round(100*(sub['WARP_62']>+0.30).sum()/n,1),
        'XLEVEL only (%)' : round(100*(
            (sub['XLEVEL'].abs()>0.80) &
            (sub['WARP_62'].abs()<=0.30)
        ).sum()/n, 1),
        'P1 Flip Rate (%)': [r['Flip Rate (%)']
                              for r in results_P1
                              if r['Period']==label][0],
        'P2 Flip Rate (%)': [r['Flip Rate (%)']
                              for r in results_P2
                              if r['Period']==label][0],
        'WARP Tail Prob'  : round(
            (sub['WARP_62'].abs()>0.30).mean(), 4),
    })

t4 = pd.DataFrame(comp_rows)
display(t4)

print('\nMechanistic correlations (monthly, P2-NN):')
print(f'  Flip rate vs WARP tail probability : r = {corr_warp:.3f}')
print(f'  Flip rate vs defect rate           : r = {corr_dr:.3f}')

In [ ]:
# ============================================================
# 9.5 — Four-Panel Summary Figure (Paper Ready)
# ============================================================
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# ---- Panel A: Defect Rate & WARP Tail Over Time ----
ax = axes[0, 0]
ax2a = ax.twinx()

monthly_dr   = df.groupby('date')['defect'].mean() * 100
monthly_warp = df.groupby('date').apply(
    lambda g: (g['WARP_62'].abs() > 0.30).mean() * 100
)

ax.bar(monthly_dr.index, monthly_dr.values,
       width=20, alpha=0.4, color='tomato',
       label='Defect Rate (%)')
ax2a.plot(monthly_warp.index, monthly_warp.values,
          color='steelblue', linewidth=2,
          marker='o', markersize=3,
          label='WARP Tail Prob (%)')

ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=4))
plt.setp(ax.get_xticklabels(), rotation=30, fontsize=8)
ax.set_title('(A) Defect Rate & WARP Tail Probability Over Time',
             fontsize=10)
ax.set_ylabel('Defect Rate (%)', color='tomato')
ax2a.set_ylabel('WARP Tail Prob (%)', color='steelblue')

lines1, l1 = ax.get_legend_handles_labels()
lines2, l2 = ax2a.get_legend_handles_labels()
ax.legend(lines1+lines2, l1+l2, fontsize=8, loc='upper left')

# ---- Panel B: Defect Composition Shift ----
ax = axes[0, 1]
years      = [2014, 2015, 2016]
warp_neg   = []
warp_pos   = []
xlev_only  = []
other_def  = []

for yr in years:
    sub = df[(df['year']==yr) & (df['defect']==1)]
    n   = len(sub)
    wn  = 100*(sub['WARP_62'] < -0.30).sum()/n
    wp  = 100*(sub['WARP_62'] >  0.30).sum()/n
    xl  = 100*((sub['XLEVEL'].abs()>0.80) &
               (sub['WARP_62'].abs()<=0.30)).sum()/n
    ot  = 100 - wn - wp - xl
    warp_neg.append(wn)
    warp_pos.append(wp)
    xlev_only.append(xl)
    other_def.append(ot)

x = np.arange(len(years))
w = 0.2
ax.bar(x - 1.5*w, warp_neg,  w, label='WARP<-0.30',
       color='steelblue', edgecolor='white', alpha=0.85)
ax.bar(x - 0.5*w, warp_pos,  w, label='WARP>+0.30',
       color='tomato', edgecolor='white', alpha=0.85)
ax.bar(x + 0.5*w, xlev_only, w, label='XLEVEL only',
       color='seagreen', edgecolor='white', alpha=0.85)
ax.bar(x + 1.5*w, other_def, w, label='Other',
       color='grey', edgecolor='white', alpha=0.85)

ax.set_title('(B) Defect Composition Shift by Year', fontsize=10)
ax.set_ylabel('% of Defective Samples')
ax.set_xticks(x)
ax.set_xticklabels(years)
ax.legend(fontsize=8)
ax.set_ylim(0, 75)

# Add n labels
for yr_i, yr in enumerate(years):
    sub = df[(df['year']==yr) & (df['defect']==1)]
    ax.text(x[yr_i], 68, f'n={len(sub):,}',
            ha='center', fontsize=8, color='black')

# ---- Panel C: MI Ranking ----
ax = axes[1, 0]
mi_sorted = mi_df.sort_values('MI', ascending=False)
bars = ax.bar(mi_sorted['Feature'], mi_sorted['MI'],
              color='steelblue', edgecolor='white', alpha=0.85)
ax.set_title('(C) Mutual Information with Defect Status',
             fontsize=10)
ax.set_ylabel('MI (nats)')
ax.set_xlabel('')
plt.setp(ax.get_xticklabels(), rotation=20, fontsize=9)

# Annotate bars
for bar, val in zip(bars, mi_sorted['MI']):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.001,
            f'{val:.4f}', ha='center', fontsize=8)

# ---- Panel D: CF Flip Rate Decay ----
ax = axes[1, 1]
p1_rates = [r['Flip Rate (%)'] for r in results_P1]
p2_rates = [r['Flip Rate (%)'] for r in results_P2]
period_labels = [r['Period'] for r in results_P1]

ax.plot(period_labels, p1_rates,
        marker='o', linewidth=2.5, markersize=9,
        color='tomato', label='P1: Mean Delta',
        linestyle='--', alpha=0.8)
ax.plot(period_labels, p2_rates,
        marker='s', linewidth=2.5, markersize=9,
        color='steelblue', label='P2: NN-Matched Delta')

# Annotate
for i, (r1, r2) in enumerate(zip(p1_rates, p2_rates)):
    ax.annotate(f'{r1:.1f}%',
                (period_labels[i], r1),
                textcoords='offset points',
                xytext=(-20, -18), fontsize=9, color='tomato')
    ax.annotate(f'{r2:.1f}%',
                (period_labels[i], r2),
                textcoords='offset points',
                xytext=(5, 8), fontsize=9, color='steelblue')

ax.set_title('(D) CF Flip Rate Decay Under Distributional Drift\n'
             '(Reference Regime: 2014)', fontsize=10)
ax.set_ylabel('Flip Rate (%)')
ax.set_ylim(0, 110)
ax.legend(fontsize=9)

# Shade reference regime
ax.axvspan(-0.4, 0.4, alpha=0.08, color='green',
           label='Reference regime')
ax.text(0, 105, 'Reference\nRegime',
        ha='center', fontsize=8, color='green')

plt.suptitle(
    'Temporal Stability of Counterfactual Explanations Under '
    'Distributional Drift\n'
    'Railway Track Geometry — Class 7 — 2013 to 2016',
    fontsize=12, y=1.01
)
plt.tight_layout()
plt.savefig('paper1_summary_figure.png', dpi=300,
            bbox_inches='tight')
plt.show()
print('Figure saved: paper1_summary_figure.png')

In [ ]:
# ============================================================
# 9.6 — Paper 1 Key Findings Summary
# ============================================================
print('=' * 65)
print('PAPER 1 — KEY FINDINGS SUMMARY')
print('Temporal Stability of Counterfactual Explanations')
print('Under Distributional Drift in a Safety-Critical')
print('Railway Cyber-Physical System')
print('=' * 65)

yr_stats = df.groupby('year').agg(
    defect_rate = ('defect','mean'),
    warp_tail   = ('WARP_62', lambda x: (x.abs()>0.30).mean()),
    warp_std    = ('WARP_62','std')
).round(4)

print('\n[F1] DISTRIBUTIONAL DRIFT IS REAL AND MEASURABLE')
print(f'  Defect rate  : 2013={yr_stats.loc[2013,"defect_rate"]:.3f} → '
      f'2016={yr_stats.loc[2016,"defect_rate"]:.3f} '
      f'(+{100*(yr_stats.loc[2016,"defect_rate"]/yr_stats.loc[2013,"defect_rate"]-1):.0f}%)')
print(f'  WARP std     : 2013={yr_stats.loc[2013,"warp_std"]:.4f} → '
      f'2016={yr_stats.loc[2016,"warp_std"]:.4f}')
print(f'  WARP tail    : 2013={yr_stats.loc[2013,"warp_tail"]:.4f} → '
      f'2016={yr_stats.loc[2016,"warp_tail"]:.4f}')

print('\n[F2] WARP IS THE DOMINANT STRUCTURAL DRIVER')
print(f'  MI(WARP; Defect)  = {mi_df[mi_df.Feature=="WARP_62"]["MI"].values[0]:.4f}')
print(f'  MI(XLEVEL; Defect)= {mi_df[mi_df.Feature=="XLEVEL"]["MI"].values[0]:.4f}')
print(f'  Unique WARP info  = {pid["Unique WARP"]:.4f} nats')
print(f'  Unique XLEVEL     = {pid["Unique XLEVEL"]:.4f} nats')
print(f'  Synergy           = {pid["Synergy"]:.4f} (no nonlinear interaction)')

print('\n[F3] MINIMAL CFs ARE FEASIBLE IN REFERENCE REGIME')
print(f'  2014 defective samples  : 649')
print(f'  CFs generated           : 637  (98.2%)')
print(f'  FRA-compliant           : 637  (100.0%)')
print(f'  Mean WARP adjustment    : '
      f'{cf_df["d_WARP_62"].abs().mean():.4f} in')
print(f'  12 failures = XLEVEL-boundary cases '
      f'(no FRA-compliant CF exists)')

print('\n[F4] CF VALIDITY DEGRADES MONOTONICALLY UNDER DRIFT')
for r1, r2 in zip(results_P1, results_P2):
    print(f'  {r1["Period"]:<12}: '
          f'P1={r1["Flip Rate (%)"]:.1f}%  '
          f'P2={r2["Flip Rate (%)"]:.1f}%')

print('\n[F5] THREE MECHANISMS OF CF FAILURE IDENTIFIED')
print('  (i)  Magnitude insufficiency: delta too small for')
print('       larger 2015/2016 WARP violations')
print('       (mean excess 2014=0.044 → 2016=0.127)')
print('  (ii) Defect regime shift: XLEVEL violations grow')
print('       from 21% (2014) to 44% (2016) of defects')
print('       WARP-based delta cannot flip XLEVEL defects')
print('  (iii)Novel defect instances: NN distance of failed')
print('       samples grows from 3.00 (2015) to 4.66 (2016)')
print('       — no close 2014 analogue exists for these cases')

print('\n[F6] MECHANISTIC LINK: DRIFT → CF FAILURE')
print(f'  Pearson r (flip rate vs WARP tail): {corr_warp:.3f}')
print(f'  Pearson r (flip rate vs defect rate): {corr_dr:.3f}')
print('  CF validity is a distribution-relative property.')
print('  Validity degrades as tail drift amplifies,')
print('  independent of transfer protocol used.')

print('\n[IMPLICATION FOR PAPER 2]')
print('  Robust CF must:')
print('  (i)  Cover full anticipated WARP violation range')
print('  (ii) Handle XLEVEL-dominated defect regime')
print('  (iii)Remain valid across the drift trajectory')
print('  → Min-max robust formulation over drift regimes')
print('=' * 65)

In [ ]:
import matplotlib.dates as mdates

monthly_nn = []
for date, grp in df[df['year'].isin([2014,2015,2016])].groupby('date'):
    # Ensure transfer_nn_delta function is defined and X_ref_orig, X_ref_delta are available
    fr, n_flip, n_def, _ = transfer_nn_delta(
        model, grp, X_ref_orig, X_ref_delta, FEATURES)
    warp_tail   = (grp['WARP_62'].abs() > 0.30).mean()
    defect_rate = grp['defect'].mean()
    monthly_nn.append({
        'date'        : date,
        'flip_rate'   : fr,
        'n_def'       : n_def,
        'warp_tail'   : warp_tail,
        'defect_rate' : defect_rate,
    })

mf_df = pd.DataFrame(monthly_nn).sort_values('date')

# Correlations
corr_warp = mf_df['flip_rate'].corr(mf_df['warp_tail'])
corr_dr   = mf_df['flip_rate'].corr(mf_df['defect_rate'])
corr_ndef = mf_df['flip_rate'].corr(mf_df['n_def'])

print('Monthly flip rate data and correlations calculated.')
print(f'  Pearson r (flip rate vs WARP tail): {corr_warp:.3f}')
print(f'  Pearson r (flip rate vs defect rate): {corr_dr:.3f}')
print(f'  Pearson r (flip rate vs n_defective): {corr_ndef:.3f}')

In [ ]:
# ============================================================
# SECTION 9 — Summary Tables & Paper-Ready Figures
# ============================================================
# Consolidates all results into publication-ready format.
# Four tables + one four-panel figure covering the full
# Paper 1 narrative:
#   Data → Drift → Information Structure → CF Degradation
# ============================================================

# ============================================================
# 9.1 — Table 1: Dataset Overview
# ============================================================
print('=' * 60)
print('TABLE 1: Dataset Overview by Year')
print('=' * 60)

t1 = df.groupby('year').agg(
    N            = ('defect', 'count'),
    N_defective  = ('defect', 'sum'),
    Defect_rate  = ('defect', 'mean'),
    WARP_std     = ('WARP_62', 'std'),
    WARP_99th    = ('WARP_62', lambda x: x.abs().quantile(0.99)),
    XLEVEL_std   = ('XLEVEL', 'std'),
    WARP_tail    = ('WARP_62', lambda x: (x.abs() > 0.30).mean()),
).round(4)

t1.columns = ['N', 'N Defective', 'Defect Rate',
              'WARP Std', 'WARP 99th |abs|',
              'XLEVEL Std', 'WARP Tail Prob']
display(t1)

# ============================================================
# 9.2 — Table 2: Information-Theoretic Structure
# ============================================================
print('\n' + '=' * 60)
print('TABLE 2: Mutual Information & PID Decomposition')
print('=' * 60)

t2_mi = mi_df[['Feature', 'MI', 'Relative']].copy()
t2_mi.columns = ['Feature', 'MI (nats)', 'Relative to WARP']
display(t2_mi.round(4))

print('\nPID Decomposition (WARP, XLEVEL → Defect):')
pid_table = pd.DataFrame({
    'Component'  : ['Redundant', 'Unique WARP',
                    'Unique XLEVEL', 'Synergy'],
    'MI (nats)'  : [pid['Redundant'], pid['Unique WARP'],
                    pid['Unique XLEVEL'], pid['Synergy']],
    '% of Total' : [
        100*pid['Redundant']/sum(pid.values()),
        100*pid['Unique WARP']/sum(pid.values()),
        100*pid['Unique XLEVEL']/sum(pid.values()),
        100*pid['Synergy']/sum(pid.values()),
    ]
}).round(4)
display(pid_table)

# ============================================================
# 9.3 — Table 3: CF Generation Summary (2014 Reference)
# ============================================================
print('\n' + '=' * 60)
print('TABLE 3: CF Generation Summary — 2014 Reference Regime')
print('=' * 60)

t3 = pd.DataFrame({
    'Metric': [
        'Total 2014 defective samples',
        'CFs successfully generated',
        'CFs failed (XLEVEL-only boundary)',
        'Success rate',
        'FRA-compliant CFs',
        'FRA compliance rate',
        'Mean CF distance (MAD-scaled L1)',
        'Mean |Δ| WARP',
        'Mean |Δ| XLEVEL',
        'Mean |Δ| GAGE',
        'Mean |Δ| ALI_L_62',
        'Mean |Δ| ALI_R_62',
    ],
    'Value': [
        649,
        637,
        12,
        '98.2%',
        637,
        '100.0%',
        f"{cf_df['cf_dist'].mean():.4f}",
        f"{cf_df['d_WARP_62'].abs().mean():.5f}",
        f"{cf_df['d_XLEVEL'].abs().mean():.5f}",
        f"{cf_df['d_GAGE'].abs().mean():.5f}",
        f"{cf_df['d_ALI_L_62'].abs().mean():.5f}",
        f"{cf_df['d_ALI_R_62'].abs().mean():.5f}",
    ]
})
display(t3)

# ============================================================
# 9.4 — Table 4: Temporal Transfer Results
# ============================================================
print('\n' + '=' * 60)
print('TABLE 4: Temporal CF Transferability')
print('=' * 60)

# Defect composition per year
comp_rows = []
for yr, label in [(2014,'Late 2014'),(2015,'2015'),(2016,'2016')]:
    sub = df[(df['year']==yr) & (df['defect']==1)]
    n   = len(sub)
    comp_rows.append({
        'Period'          : label,
        'N Defective'     : n,
        'WARP<-0.30 (%)'  : round(100*(sub['WARP_62']<-0.30).sum()/n,1),
        'WARP>+0.30 (%)'  : round(100*(sub['WARP_62']>+0.30).sum()/n,1),
        'XLEVEL only (%)' : round(100*(
            (sub['XLEVEL'].abs()>0.80) &
            (sub['WARP_62'].abs()<=0.30)
        ).sum()/n, 1),
        'P1 Flip Rate (%)': [r['Flip Rate (%)']
                              for r in results_P1
                              if r['Period']==label][0],
        'P2 Flip Rate (%)': [r['Flip Rate (%)']
                              for r in results_P2
                              if r['Period']==label][0],
        'WARP Tail Prob'  : round(
            (sub['WARP_62'].abs()>0.30).mean(), 4),
    })

t4 = pd.DataFrame(comp_rows)
display(t4)

print('\nMechanistic correlations (monthly, P2-NN):')
print(f'  Flip rate vs WARP tail probability : r = {corr_warp:.3f}')
print(f'  Flip rate vs defect rate           : r = {corr_dr:.3f}')

In [ ]:
# ============================================================
# 9.5 — Four-Panel Summary Figure (Paper Ready)
# ============================================================
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# ---- Panel A: Defect Rate & WARP Tail Over Time ----
ax = axes[0, 0]
ax2a = ax.twinx()

monthly_dr   = df.groupby('date')['defect'].mean() * 100
monthly_warp = df.groupby('date').apply(
    lambda g: (g['WARP_62'].abs() > 0.30).mean() * 100
)

ax.bar(monthly_dr.index, monthly_dr.values,
       width=20, alpha=0.4, color='tomato',
       label='Defect Rate (%)')
ax2a.plot(monthly_warp.index, monthly_warp.values,
          color='steelblue', linewidth=2,
          marker='o', markersize=3,
          label='WARP Tail Prob (%)')

ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=4))
plt.setp(ax.get_xticklabels(), rotation=30, fontsize=8)
ax.set_title('(A) Defect Rate & WARP Tail Probability Over Time',
             fontsize=10)
ax.set_ylabel('Defect Rate (%)', color='tomato')
ax2a.set_ylabel('WARP Tail Prob (%)', color='steelblue')

lines1, l1 = ax.get_legend_handles_labels()
lines2, l2 = ax2a.get_legend_handles_labels()
ax.legend(lines1+lines2, l1+l2, fontsize=8, loc='upper left')

# ---- Panel B: Defect Composition Shift ----
ax = axes[0, 1]
years      = [2014, 2015, 2016]
warp_neg   = []
warp_pos   = []
xlev_only  = []
other_def  = []

for yr in years:
    sub = df[(df['year']==yr) & (df['defect']==1)]
    n   = len(sub)
    wn  = 100*(sub['WARP_62'] < -0.30).sum()/n
    wp  = 100*(sub['WARP_62'] >  0.30).sum()/n
    xl  = 100*((sub['XLEVEL'].abs()>0.80) &
               (sub['WARP_62'].abs()<=0.30)).sum()/n
    ot  = 100 - wn - wp - xl
    warp_neg.append(wn)
    warp_pos.append(wp)
    xlev_only.append(xl)
    other_def.append(ot)

x = np.arange(len(years))
w = 0.2
ax.bar(x - 1.5*w, warp_neg,  w, label='WARP<-0.30',
       color='steelblue', edgecolor='white', alpha=0.85)
ax.bar(x - 0.5*w, warp_pos,  w, label='WARP>+0.30',
       color='tomato', edgecolor='white', alpha=0.85)
ax.bar(x + 0.5*w, xlev_only, w, label='XLEVEL only',
       color='seagreen', edgecolor='white', alpha=0.85)
ax.bar(x + 1.5*w, other_def, w, label='Other',
       color='grey', edgecolor='white', alpha=0.85)

ax.set_title('(B) Defect Composition Shift by Year', fontsize=10)
ax.set_ylabel('% of Defective Samples')
ax.set_xticks(x)
ax.set_xticklabels(years)
ax.legend(fontsize=8)
ax.set_ylim(0, 75)

# Add n labels
for yr_i, yr in enumerate(years):
    sub = df[(df['year']==yr) & (df['defect']==1)]
    ax.text(x[yr_i], 68, f'n={len(sub):,}',
            ha='center', fontsize=8, color='black')

# ---- Panel C: MI Ranking ----
ax = axes[1, 0]
mi_sorted = mi_df.sort_values('MI', ascending=False)
bars = ax.bar(mi_sorted['Feature'], mi_sorted['MI'],
              color='steelblue', edgecolor='white', alpha=0.85)
ax.set_title('(C) Mutual Information with Defect Status',
             fontsize=10)
ax.set_ylabel('MI (nats)')
ax.set_xlabel('')
plt.setp(ax.get_xticklabels(), rotation=20, fontsize=9)

# Annotate bars
for bar, val in zip(bars, mi_sorted['MI']):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.001,
            f'{val:.4f}', ha='center', fontsize=8)

# ---- Panel D: CF Flip Rate Decay ----
ax = axes[1, 1]
p1_rates = [r['Flip Rate (%)'] for r in results_P1]
p2_rates = [r['Flip Rate (%)'] for r in results_P2]
period_labels = [r['Period'] for r in results_P1]

ax.plot(period_labels, p1_rates,
        marker='o', linewidth=2.5, markersize=9,
        color='tomato', label='P1: Mean Delta',
        linestyle='--', alpha=0.8)
ax.plot(period_labels, p2_rates,
        marker='s', linewidth=2.5, markersize=9,
        color='steelblue', label='P2: NN-Matched Delta')

# Annotate
for i, (r1, r2) in enumerate(zip(p1_rates, p2_rates)):
    ax.annotate(f'{r1:.1f}%',
                (period_labels[i], r1),
                textcoords='offset points',
                xytext=(-20, -18), fontsize=9, color='tomato')
    ax.annotate(f'{r2:.1f}%',
                (period_labels[i], r2),
                textcoords='offset points',
                xytext=(5, 8), fontsize=9, color='steelblue')

ax.set_title('(D) CF Flip Rate Decay Under Distributional Drift\n'
             '(Reference Regime: 2014)', fontsize=10)
ax.set_ylabel('Flip Rate (%)')
ax.set_ylim(0, 110)
ax.legend(fontsize=9)

# Shade reference regime
ax.axvspan(-0.4, 0.4, alpha=0.08, color='green',
           label='Reference regime')
ax.text(0, 105, 'Reference\nRegime',
        ha='center', fontsize=8, color='green')

plt.suptitle(
    'Temporal Stability of Counterfactual Explanations Under '
    'Distributional Drift\n'
    'Railway Track Geometry — Class 7 — 2013 to 2016',
    fontsize=12, y=1.01
)
plt.tight_layout()
plt.savefig('paper1_summary_figure.png', dpi=300,
            bbox_inches='tight')
plt.show()
print('Figure saved: paper1_summary_figure.png')

In [ ]:
# ============================================================
# 9.6 — Paper 1 Key Findings Summary
# ============================================================
print('=' * 65)
print('PAPER 1 — KEY FINDINGS SUMMARY')
print('Temporal Stability of Counterfactual Explanations')
print('Under Distributional Drift in a Safety-Critical')
print('Railway Cyber-Physical System')
print('=' * 65)

yr_stats = df.groupby('year').agg(
    defect_rate = ('defect','mean'),
    warp_tail   = ('WARP_62', lambda x: (x.abs()>0.30).mean()),
    warp_std    = ('WARP_62','std')
).round(4)

print('\n[F1] DISTRIBUTIONAL DRIFT IS REAL AND MEASURABLE')
print(f'  Defect rate  : 2013={yr_stats.loc[2013,"defect_rate"]:.3f} → '
      f'2016={yr_stats.loc[2016,"defect_rate"]:.3f} '
      f'(+{100*(yr_stats.loc[2016,"defect_rate"]/yr_stats.loc[2013,"defect_rate"]-1):.0f}%)')
print(f'  WARP std     : 2013={yr_stats.loc[2013,"warp_std"]:.4f} → '
      f'2016={yr_stats.loc[2016,"warp_std"]:.4f}')
print(f'  WARP tail    : 2013={yr_stats.loc[2013,"warp_tail"]:.4f} → '
      f'2016={yr_stats.loc[2016,"warp_tail"]:.4f}')

print('\n[F2] WARP IS THE DOMINANT STRUCTURAL DRIVER')
print(f'  MI(WARP; Defect)  = {mi_df[mi_df.Feature=="WARP_62"]["MI"].values[0]:.4f}')
print(f'  MI(XLEVEL; Defect)= {mi_df[mi_df.Feature=="XLEVEL"]["MI"].values[0]:.4f}')
print(f'  Unique WARP info  = {pid["Unique WARP"]:.4f} nats')
print(f'  Unique XLEVEL     = {pid["Unique XLEVEL"]:.4f} nats')
print(f'  Synergy           = {pid["Synergy"]:.4f} (no nonlinear interaction)')

print('\n[F3] MINIMAL CFs ARE FEASIBLE IN REFERENCE REGIME')
print(f'  2014 defective samples  : 649')
print(f'  CFs generated           : 637  (98.2%)')
print(f'  FRA-compliant           : 637  (100.0%)')
print(f'  Mean WARP adjustment    : '
      f'{cf_df["d_WARP_62"].abs().mean():.4f} in')
print(f'  12 failures = XLEVEL-boundary cases '
      f'(no FRA-compliant CF exists)')

print('\n[F4] CF VALIDITY DEGRADES MONOTONICALLY UNDER DRIFT')
for r1, r2 in zip(results_P1, results_P2):
    print(f'  {r1["Period"]:<12}: '
          f'P1={r1["Flip Rate (%)"]:.1f}%  '
          f'P2={r2["Flip Rate (%)"]:.1f}%')

print('\n[F5] THREE MECHANISMS OF CF FAILURE IDENTIFIED')
print('  (i)  Magnitude insufficiency: delta too small for')
print('       larger 2015/2016 WARP violations')
print('       (mean excess 2014=0.044 → 2016=0.127)')
print('  (ii) Defect regime shift: XLEVEL violations grow')
print('       from 21% (2014) to 44% (2016) of defects')
print('       WARP-based delta cannot flip XLEVEL defects')
print('  (iii)Novel defect instances: NN distance of failed')
print('       samples grows from 3.00 (2015) to 4.66 (2016)')
print('       — no close 2014 analogue exists for these cases')

print('\n[F6] MECHANISTIC LINK: DRIFT → CF FAILURE')
print(f'  Pearson r (flip rate vs WARP tail): {corr_warp:.3f}')
print(f'  Pearson r (flip rate vs defect rate): {corr_dr:.3f}')
print('  CF validity is a distribution-relative property.')
print('  Validity degrades as tail drift amplifies,')
print('  independent of transfer protocol used.')

print('\n[IMPLICATION FOR PAPER 2]')
print('  Robust CF must:')
print('  (i)  Cover full anticipated WARP violation range')
print('  (ii) Handle XLEVEL-dominated defect regime')
print('  (iii)Remain valid across the drift trajectory')
print('  → Min-max robust formulation over drift regimes')
print('=' * 65)

---
## Section 8 — Temporal Transfer & Flip Rate Decay

The core empirical contribution of Paper 1.

**Protocol:**
1. For each 2014 defective sample, compute its minimal CF delta (done in Section 7)
2. For each year (late 2014, 2015, 2016), take ALL defective samples
3. Apply the **mean** 2014 CF delta to each defective sample
4. Check whether the shifted sample flips to non-defective under the model
5. Report flip rate by year — degradation = counterfactual instability under drift

**Flip rate** = fraction of defective samples successfully flipped to non-defective by the reference-regime CF.

In [ ]:
# ============================================================
# 8.1 — Compute Mean Reference Delta from 2014 CFs
# ============================================================
delta_cols  = [f'd_{f}' for f in FEATURES]
mean_delta  = cf_df[delta_cols].mean().values
median_delta = cf_df[delta_cols].median().values

print('Mean 2014 CF delta:')
for f, d in zip(FEATURES, mean_delta):
    print(f'  {f:<12}: {d:+.5f}')

In [ ]:
# ============================================================
# 8.2 — Temporal Transfer: Flip Rate by Year
# ============================================================
def compute_flip_rate(model, year_df, delta, features, bounds):
    """Apply a fixed delta to all defective samples; return flip rate."""
    defective = year_df[year_df['defect'] == 1].copy()
    if len(defective) == 0:
        return 0.0, 0, 0

    X_orig = defective[features].values
    X_cf   = X_orig + delta

    # Clip to bounds
    for j, f in enumerate(features):
        lo, hi = bounds[f]
        X_cf[:, j] = np.clip(X_cf[:, j], lo, hi)

    pred_orig = model.predict(X_orig)
    pred_cf   = model.predict(X_cf)

    n_def     = (pred_orig == 1).sum()
    n_flipped = ((pred_orig == 1) & (pred_cf == 0)).sum()
    flip_rate = n_flipped / max(1, n_def)
    return flip_rate, n_flipped, n_def


periods = [
    ('Late 2014', df[(df['year'] == 2014)]),
    ('2015',      df[df['year'] == 2015]),
    ('2016',      df[df['year'] == 2016]),
]

transfer_results = []
for label, period_df in periods:
    fr, n_flip, n_def = compute_flip_rate(model, period_df, mean_delta, FEATURES, BOUNDS)
    transfer_results.append({
        'Period'        : label,
        'Defects'       : n_def,
        'Flipped'       : n_flip,
        'Not Flipped'   : n_def - n_flip,
        'Flip Rate (%)'  : round(100 * fr, 2),
    })

transfer_df = pd.DataFrame(transfer_results)
display(transfer_df)

In [ ]:
# ============================================================
# 8.3 — Flip Rate Decay Plot (Key Paper Figure)
# ============================================================
fig, ax1 = plt.subplots(figsize=(9, 5))

periods_label = transfer_df['Period'].tolist()
flip_rates    = transfer_df['Flip Rate (%)'].tolist()

ax1.plot(periods_label, flip_rates, marker='o', linewidth=2.5,
         color='tomato', markersize=9, label='Flip Rate (%)')
ax1.set_ylabel('Flip Rate (%)', color='tomato')
ax1.tick_params(axis='y', labelcolor='tomato')
ax1.set_ylim(0, 100)

# Overlay yearly defect rate on secondary axis
ax2 = ax1.twinx()
yearly_dr = df.groupby('year')['defect'].mean() * 100
# Map years to period labels for overlay
dr_vals = [yearly_dr.get(2014, 0), yearly_dr.get(2015, 0), yearly_dr.get(2016, 0)]
ax2.bar(periods_label, dr_vals, alpha=0.25, color='steelblue', label='Defect Rate (%)')
ax2.set_ylabel('Defect Rate (%)', color='steelblue')
ax2.tick_params(axis='y', labelcolor='steelblue')

ax1.set_title('Counterfactual Flip Rate Decay Under Distributional Drift\n(Reference CF: 2014)', fontsize=12)
ax1.set_xlabel('Time Period')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='center right')

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 8.4 — Monthly Flip Rate (Higher Resolution)
#        Links flip rate decay to monthly drift metrics
# ============================================================
monthly_flip = []
for date, grp in df[df['year'].isin([2014, 2015, 2016])].groupby('date'):
    fr, n_flip, n_def = compute_flip_rate(model, grp, mean_delta, FEATURES, BOUNDS)
    warp_tail = (grp['WARP_62'].abs() > 0.30).mean()
    monthly_flip.append({
        'date'      : date,
        'flip_rate' : fr,
        'n_def'     : n_def,
        'warp_tail' : warp_tail,
    })

mf_df = pd.DataFrame(monthly_flip).sort_values('date')

fig, ax1 = plt.subplots(figsize=(12, 5))
ax1.plot(mf_df['date'], mf_df['flip_rate']*100, marker='o',
         color='tomato', linewidth=2, label='Monthly Flip Rate (%)')
ax1.set_ylabel('Flip Rate (%)', color='tomato')
ax1.tick_params(axis='y', labelcolor='tomato')

ax2 = ax1.twinx()
ax2.plot(mf_df['date'], mf_df['warp_tail'], marker='s',
         color='steelblue', linewidth=2, linestyle='--', label='WARP Tail Prob')
ax2.set_ylabel('WARP Tail Probability', color='steelblue')
ax2.tick_params(axis='y', labelcolor='steelblue')

ax1.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax1.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
plt.xticks(rotation=45)
ax1.set_title('Monthly Flip Rate vs WARP Tail Probability')
ax1.set_xlabel('Date')

lines1, l1 = ax1.get_legend_handles_labels()
lines2, l2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, l1 + l2)
plt.tight_layout()
plt.show()

# Correlation between flip rate and tail probability
corr_val = mf_df['flip_rate'].corr(mf_df['warp_tail'])
print(f'Pearson correlation (flip rate vs WARP tail prob): {corr_val:.3f}')

---
## Section 9 — Summary Tables & Paper-Ready Plots

Consolidated results for Paper 1.

In [ ]:
# ============================================================
# 9.1 — Master Results Table
# ============================================================
print('=== TABLE 1: Dataset Overview ===')
ds_summary = df.groupby('year').agg(
    N        = ('defect', 'count'),
    Defects  = ('defect', 'sum'),
    DefRate  = ('defect', 'mean'),
    WARP_std = ('WARP_62', 'std'),
    WARP_99  = ('WARP_62', lambda x: x.abs().quantile(0.99)),
).round(4)
display(ds_summary)

print('\n=== TABLE 2: Counterfactual Transferability ===')
display(transfer_df)

print('\n=== TABLE 3: MI Ranking ===')
display(mi_df[['Feature','MI','Relative']].round(4))

print('\n=== TABLE 4: PID Decomposition ===')
pid_df = pd.DataFrame([pid]).T.rename(columns={0:'MI'})
pid_df['%'] = (100 * pid_df['MI'] / pid_df['MI'].sum()).round(1)
display(pid_df.round(4))

In [ ]:
# ============================================================
# 9.2 — Four-Panel Summary Figure (Paper Ready)
# ============================================================
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

# Panel A: Yearly defect rate
ax = axes[0, 0]
yr_dr = df.groupby('year')['defect'].mean() * 100
ax.bar(yr_dr.index, yr_dr.values, color='tomato', edgecolor='white')
ax.set_title('(A) Yearly Defect Rate')
ax.set_ylabel('Defect Rate (%)')
ax.set_xlabel('Year')

# Panel B: WARP tail drift
ax = axes[0, 1]
ax.plot(monthly_tail['date'], monthly_tail['WARP_tail'],
        color='steelblue', linewidth=2, marker='o', markersize=3)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=4))
plt.setp(ax.get_xticklabels(), rotation=30, fontsize=8)
ax.set_title('(B) WARP Tail Probability Over Time')
ax.set_ylabel('P(|WARP| > 0.30)')

# Panel C: MI bar chart
ax = axes[1, 0]
sns.barplot(data=mi_df, x='Feature', y='MI', palette='Blues_d', ax=ax)
ax.set_title('(C) Mutual Information with Defect')
ax.set_ylabel('MI (nats)')
ax.set_xlabel('')
plt.setp(ax.get_xticklabels(), rotation=20)

# Panel D: Flip rate decay
ax = axes[1, 1]
ax.plot(transfer_df['Period'], transfer_df['Flip Rate (%)'],
        marker='o', linewidth=2.5, color='tomato', markersize=9)
ax.set_title('(D) CF Flip Rate Decay Over Time')
ax.set_ylabel('Flip Rate (%)')
ax.set_ylim(0, 100)
for _, row in transfer_df.iterrows():
    ax.annotate(f"{row['Flip Rate (%)']:.1f}%",
                (row['Period'], row['Flip Rate (%)']),
                textcoords='offset points', xytext=(0, 10), ha='center', fontsize=9)

plt.suptitle('Temporal Stability of Counterfactual Explanations Under Drift\n(Railway Track Geometry, 2013–2016)',
             fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

---
## End of Paper 1 Notebook

**Key Findings:**
1. Distributional drift is real and measurable — WARP variance and tail probability increase steadily 2013→2016
2. WARP is the dominant structural driver of defect risk (MI, PID, XGBoost importance all agree)
3. Minimal CFs computed in the reference regime (2014) degrade over time — flip rate drops from ~68% to ~21%
4. The decay is mechanistically linked to WARP tail amplification
5. Counterfactual validity is a distribution-relative property, not a fixed attribute of the explanation

**Paper 2 (Robust CF):** The min-max robust formulation and population-level robust CF evaluation will be built in a separate notebook.

In [ ]:
# ============================================================
# SECTION 10 — Ground-Truth Proxy Validation via Future Persistence
# Uses repeated monthly inspections of the same segment
# to test whether CF severity predicts near-term persistence
# better than standard baselines (distance-to-limit, local TQI).
# ============================================================

import numpy as np
import pandas as pd

from scipy.stats import spearmanr
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, brier_score_loss
from sklearn.calibration import calibration_curve

# ------------------------------------------------------------
# 10.1 — Basic setup / defensive copy
# ------------------------------------------------------------
df_val = df.copy()
df_val['date'] = pd.to_datetime(df_val['date'])

cf_val = cf_df.copy()
cf_val['date_orig'] = pd.to_datetime(cf_val['date_orig'])

# Keep only the 2014 defective samples for which a CF exists
cf_val = cf_val.sort_values(['MILE', 'FEET', 'date_orig']).reset_index(drop=True)

print('=' * 70)
print('GROUND-TRUTH PROXY VALIDATION SETUP')
print('=' * 70)
print(f'CF-supported 2014 defective samples: {len(cf_val):,}')


# ------------------------------------------------------------
# 10.2 — Helper functions
# ------------------------------------------------------------
def compute_distance_to_limit_row(row, project_limits):
    """
    For a defective sample, compute the largest exceedance above the
    project-limit envelope across all tracked features.

    Returns 0 if the point is not beyond any project limit.
    """
    exceedances = []

    for f, (lo, hi) in project_limits.items():
        x = float(row[f])

        if f == 'GAGE':
            if x < lo:
                exceedances.append(lo - x)
            elif x > hi:
                exceedances.append(x - hi)
            else:
                exceedances.append(0.0)
        else:
            # symmetric limit around zero in your current setup
            exceedances.append(max(0.0, abs(x) - hi))

    return float(max(exceedances))


def add_local_tqi(df_in):
    """
    Local TQI proxy:
    std of each feature within the same (date, MILE) group,
    then average across FEATURES.

    This gives a local, month-specific spread metric instead of
    using one corridor-wide monthly TQI value for every point.
    """
    df_out = df_in.copy()

    grp = df_out.groupby(['date', 'MILE'])

    for f in FEATURES:
        df_out[f'tqi_{f}'] = grp[f].transform('std')

    tqi_cols = [f'tqi_{f}' for f in FEATURES]
    df_out['local_tqi'] = df_out[tqi_cols].mean(axis=1)

    # If a group has only one point and std is NaN, fill from same-date global std
    date_std_map = {}
    for f in FEATURES:
        date_std_map[f] = df_out.groupby('date')[f].transform('std')

    for f in FEATURES:
        df_out[f'tqi_{f}'] = df_out[f'tqi_{f}'].fillna(date_std_map[f])

    df_out['local_tqi'] = df_out[[f'tqi_{f}' for f in FEATURES]].mean(axis=1)

    return df_out


def build_future_labels(panel_df, horizons=(1, 3, 6)):
    """
    For each segment (MILE, FEET), build future persistence labels:
      persist_1m: defective exactly 1 step ahead
      persist_3m: defective at least once in next 3 inspections
      persist_6m: defective at least once in next 6 inspections
      persist_count_3m: number of future defects in next 3 inspections
      persist_count_6m: number of future defects in next 6 inspections

    Assumes monthly observations ordered by date within segment.
    """
    out = panel_df.sort_values(['MILE', 'FEET', 'date']).copy()

    def _per_segment(g):
        g = g.sort_values('date').copy()
        defect_vals = g['defect'].astype(int).values
        n = len(g)

        for h in horizons:
            any_list = []
            count_list = []

            for i in range(n):
                future = defect_vals[i+1:i+1+h]
                if len(future) == 0:
                    any_list.append(np.nan)
                    count_list.append(np.nan)
                else:
                    any_list.append(int(np.any(future == 1)))
                    count_list.append(int(np.sum(future == 1)))

            g[f'persist_{h}m'] = any_list
            g[f'persist_count_{h}m'] = count_list

        # exact next inspection
        g['next_defect'] = list(defect_vals[1:]) + [np.nan]

        return g

    out = out.groupby(['MILE', 'FEET'], group_keys=False).apply(_per_segment)
    return out.reset_index(drop=True)


def fit_eval_logit(X, y, model_name='model'):
    """
    Fit logistic regression and return AUC, Brier, predictions, coefficients.
    """
    X = X.copy()
    y = y.copy()

    valid = X.notna().all(axis=1) & y.notna()
    Xv = X.loc[valid]
    yv = y.loc[valid].astype(int)

    if len(np.unique(yv)) < 2:
        raise ValueError(f'{model_name}: target has only one class after filtering.')

    clf = LogisticRegression(max_iter=5000, solver='liblinear')
    clf.fit(Xv, yv)

    p = clf.predict_proba(Xv)[:, 1]
    auc = float(roc_auc_score(yv, p))
    brier = float(brier_score_loss(yv, p))

    coef_df = pd.DataFrame({
        'feature': Xv.columns,
        'coef': clf.coef_[0]
    }).sort_values('coef', key=np.abs, ascending=False)

    return {
        'name': model_name,
        'n': int(len(yv)),
        'auc': auc,
        'brier': brier,
        'y': yv,
        'p': p,
        'coef_df': coef_df
    }


def make_calibration_table(y_true, p_pred, n_bins=10):
    """
    Simple calibration table for paper/reporting.
    """
    frac_pos, mean_pred = calibration_curve(y_true, p_pred, n_bins=n_bins, strategy='quantile')
    calib = pd.DataFrame({
        'bin_mean_pred': mean_pred,
        'bin_frac_positive': frac_pos
    })
    return calib


# ------------------------------------------------------------
# 10.3 — Build local TQI + future persistence panel
# ------------------------------------------------------------
df_val = add_local_tqi(df_val)
df_val = build_future_labels(df_val, horizons=(1, 3, 6))

# Merge persistence + local TQI onto the CF-supported 2014 samples
merge_cols = [
    'MILE', 'FEET', 'date',
    'defect',
    'local_tqi',
    'next_defect',
    'persist_1m', 'persist_3m', 'persist_6m',
    'persist_count_1m', 'persist_count_3m', 'persist_count_6m'
]

cf_eval = cf_val.merge(
    df_val[merge_cols],
    left_on=['MILE', 'FEET', 'date_orig'],
    right_on=['MILE', 'FEET', 'date'],
    how='left'
).drop(columns=['date'])

# ------------------------------------------------------------
# 10.4 — Add baseline severity measures
# ------------------------------------------------------------
# CF severity
cf_eval['cf_warp_abs'] = cf_eval['d_WARP_62'].abs()

# Distance-to-limit at original point
orig_cols_map = {f: f'x0_{f}' for f in FEATURES}

def _distance_from_x0(row):
    fake_row = {f: row[orig_cols_map[f]] for f in FEATURES}
    return compute_distance_to_limit_row(fake_row, PROJECT_LIMITS)

cf_eval['distance_to_limit'] = cf_eval.apply(_distance_from_x0, axis=1)

# Optional: count how many future inspections remain defective
# already created as persist_count_3m, persist_count_6m

print('\nValidation dataset summary:')
print(cf_eval[
    ['cf_dist', 'cf_warp_abs', 'distance_to_limit', 'local_tqi',
     'persist_1m', 'persist_3m', 'persist_6m',
     'persist_count_3m', 'persist_count_6m']
].describe().round(4))


# ------------------------------------------------------------
# 10.5 — Spearman correlations with persistence counts
# ------------------------------------------------------------
print('\n' + '=' * 70)
print('SPEARMAN CORRELATIONS WITH FUTURE PERSISTENCE')
print('=' * 70)

corr_rows = []
for feature_name in ['cf_dist', 'cf_warp_abs', 'distance_to_limit', 'local_tqi']:
    for outcome_name in ['persist_count_3m', 'persist_count_6m']:
        sub = cf_eval[[feature_name, outcome_name]].dropna()
        rho, pval = spearmanr(sub[feature_name], sub[outcome_name])
        corr_rows.append({
            'Predictor': feature_name,
            'Outcome': outcome_name,
            'Spearman_rho': float(rho),
            'p_value': float(pval),
            'n': int(len(sub))
        })

corr_df = pd.DataFrame(corr_rows).sort_values(['Outcome', 'Spearman_rho'], ascending=[True, False])
display(corr_df.round(4))


# ------------------------------------------------------------
# 10.6 — Nested logistic models for persist_1m
# ------------------------------------------------------------
print('\n' + '=' * 70)
print('NESTED LOGISTIC MODELS — TARGET: persist_1m')
print('=' * 70)

target = cf_eval['persist_1m']

model_specs = {
    'A_distance_only': ['distance_to_limit'],
    'B_tqi_only': ['local_tqi'],
    'C_cf_only': ['cf_dist'],
    'C2_cf_warp_only': ['cf_warp_abs'],
    'D_distance_plus_tqi': ['distance_to_limit', 'local_tqi'],
    'E_distance_tqi_cf': ['distance_to_limit', 'local_tqi', 'cf_dist'],
    'E2_distance_tqi_cfwarp': ['distance_to_limit', 'local_tqi', 'cf_warp_abs'],
}

fit_results = {}
summary_rows = []

for name, cols in model_specs.items():
    result = fit_eval_logit(cf_eval[cols], target, model_name=name)
    fit_results[name] = result
    summary_rows.append({
        'Model': name,
        'Features': ', '.join(cols),
        'n': result['n'],
        'ROC_AUC': result['auc'],
        'Brier': result['brier']
    })

model_summary_1m = pd.DataFrame(summary_rows).sort_values('ROC_AUC', ascending=False)
display(model_summary_1m.round(4))


# ------------------------------------------------------------
# 10.7 — Also evaluate 3m and 6m persistence
# ------------------------------------------------------------
print('\n' + '=' * 70)
print('NESTED LOGISTIC MODELS — TARGETS: persist_3m, persist_6m')
print('=' * 70)

all_target_summaries = []

for outcome in ['persist_3m', 'persist_6m']:
    y_out = cf_eval[outcome]
    rows = []

    for name, cols in model_specs.items():
        result = fit_eval_logit(cf_eval[cols], y_out, model_name=f'{name}_{outcome}')
        rows.append({
            'Outcome': outcome,
            'Model': name,
            'Features': ', '.join(cols),
            'n': result['n'],
            'ROC_AUC': result['auc'],
            'Brier': result['brier']
        })

    temp = pd.DataFrame(rows).sort_values(['Outcome', 'ROC_AUC'], ascending=[True, False])
    all_target_summaries.append(temp)

model_summary_all = pd.concat(all_target_summaries, ignore_index=True)
display(model_summary_all.round(4))


# ------------------------------------------------------------
# 10.8 — Calibration table for the best 1m model
# ------------------------------------------------------------
best_model_name = model_summary_1m.iloc[0]['Model']
best_res = fit_results[best_model_name]

print('\n' + '=' * 70)
print(f'CALIBRATION TABLE — BEST 1M MODEL: {best_model_name}')
print('=' * 70)

calib_df = make_calibration_table(best_res['y'], best_res['p'], n_bins=10)
display(calib_df.round(4))

print('\nCoefficients for best 1m model:')
display(best_res['coef_df'].round(4))


# ------------------------------------------------------------
# 10.9 — Optional compact results table for the paper
# ------------------------------------------------------------
paper_table = model_summary_all.copy()
paper_table['ROC_AUC'] = paper_table['ROC_AUC'].round(3)
paper_table['Brier'] = paper_table['Brier'].round(3)

print('\n' + '=' * 70)
print('PAPER-READY VALIDATION TABLE')
print('=' * 70)
display(paper_table)


# ------------------------------------------------------------
# 10.10 — Simple interpretation helper
# ------------------------------------------------------------
print('\n' + '=' * 70)
print('INTERPRETATION CHECK')
print('=' * 70)

best_3m = model_summary_all[model_summary_all['Outcome'] == 'persist_3m'].iloc[0]
best_6m = model_summary_all[model_summary_all['Outcome'] == 'persist_6m'].iloc[0]

print(f'Best model for persist_1m: {best_model_name}')
print(f'  AUC   = {float(model_summary_1m.iloc[0]["ROC_AUC"]):.4f}')
print(f'  Brier = {float(model_summary_1m.iloc[0]["Brier"]):.4f}')

print(f'\nBest model for persist_3m: {best_3m["Model"]}')
print(f'  AUC   = {float(best_3m["ROC_AUC"]):.4f}')
print(f'  Brier = {float(best_3m["Brier"]):.4f}')

print(f'\nBest model for persist_6m: {best_6m["Model"]}')
print(f'  AUC   = {float(best_6m["ROC_AUC"]):.4f}')
print(f'  Brier = {float(best_6m["Brier"]):.4f}')

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Bin CF severity
cf_eval['cf_bin'] = pd.qcut(cf_eval['cf_warp_abs'], q=10, duplicates='drop')

# Compute persistence probability per bin
plot_df = cf_eval.groupby('cf_bin')['persist_3m'].mean().reset_index()

# Plot
plt.figure()
plt.plot(range(len(plot_df)), plot_df['persist_3m'], marker='o')
plt.xlabel('CF Severity (binned)')
plt.ylabel('P(persist_3m = 1)')
plt.title('CF Severity vs Future Defect Persistence')
plt.grid(True)
plt.show()

In [ ]:
from sklearn.metrics import roc_curve

plt.figure()

for name in ['C2_cf_warp_only', 'A_distance_only', 'B_tqi_only']:
    res = fit_results[name]
    fpr, tpr, _ = roc_curve(res['y'], res['p'])
    plt.plot(fpr, tpr, label=f"{name} (AUC={res['auc']:.3f})")

plt.plot([0,1], [0,1], linestyle='--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Comparison: CF vs Baselines')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# ============================================================
# ALL VALIDATION FIGURES — A, B, ROC
# Run after Section 10 of the validation notebook.
# Requires: cf_eval, fit_results, model_summary_1m,
#           model_summary_all, corr_df
# ============================================================
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_curve

os.makedirs('figs', exist_ok=True)
sns.set_theme(style='whitegrid', font_scale=1.05)
plt.rcParams.update({'figure.dpi': 150})

# ── Shared colours ────────────────────────────────────────────
C_CF    = '#D4537E'
C_DIST  = '#EF9F27'
C_COMBO = '#185FA5'
C_TQI   = '#888780'

# ── Tier boundaries (Table VI) ───────────────────────────────
def assign_tier(w):
    if w < 0.03:   return 'Marginal\n(<0.03 in)'
    elif w < 0.08: return 'Moderate\n(0.03–0.08)'
    elif w < 0.15: return 'Severe\n(0.08–0.15)'
    else:          return 'Critical\n(>0.15 in)'

TIER_ORDER  = ['Marginal\n(<0.03 in)', 'Moderate\n(0.03–0.08)',
               'Severe\n(0.08–0.15)']
TIER_COLORS = ['#C0DD97', '#FAC775', '#F0997B']


# ============================================================
# FIGURE A — Spearman ρ (left) + AUC across horizons (right)
# ============================================================

# Pull Spearman values from corr_df
spearman_3m = (
    corr_df[corr_df['Outcome'] == 'persist_count_3m']
    .set_index('Predictor')['Spearman_rho']
)
predictors = ['cf_warp_abs', 'cf_dist', 'distance_to_limit', 'local_tqi']
labels     = ['|ΔWARP| (CF)', 'CF dist', 'Dist-to-limit', 'TQI']
colors_bar = [C_CF, C_CF, C_DIST, C_TQI]
rho_vals   = [float(spearman_3m.get(p, 0)) for p in predictors]

# Pull AUC values
def get_auc(model_name, outcome, summary_df=None, summary_1m=None):
    if outcome == 'persist_1m':
        row = summary_1m[summary_1m['Model'] == model_name]
    else:
        row = summary_df[
            (summary_df['Model']   == model_name) &
            (summary_df['Outcome'] == outcome)
        ]
    return float(row['ROC_AUC'].iloc[0]) if len(row) else np.nan

horizons = ['1-month', '3-month', '6-month']
outcomes = ['persist_1m', 'persist_3m', 'persist_6m']

auc_cf    = [get_auc('C2_cf_warp_only',      o, model_summary_all, model_summary_1m) for o in outcomes]
auc_dist  = [get_auc('A_distance_only',      o, model_summary_all, model_summary_1m) for o in outcomes]
auc_combo = [get_auc('E2_distance_tqi_cfwarp', o, model_summary_all, model_summary_1m) for o in outcomes]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# ── Left: Spearman bars ──────────────────────────────────────
ax = axes[0]
y_pos = np.arange(len(predictors))
ax.barh(y_pos, rho_vals[::-1],
        color=colors_bar[::-1], edgecolor='white', height=0.55)
ax.set_yticks(y_pos)
ax.set_yticklabels(labels[::-1], fontsize=10)
ax.set_xlim(0, 0.72)
ax.set_xlabel('Spearman ρ with 3-month persistence count', fontsize=10)
ax.set_title('(A) Predictor–persistence correlation', fontsize=10, pad=8)
ax.axvline(0, color='#B4B2A9', linewidth=0.8)

for i, (pred, rho) in enumerate(zip(predictors[::-1], rho_vals[::-1])):
    sig_row = corr_df[
        (corr_df['Predictor'] == pred) &
        (corr_df['Outcome']   == 'persist_count_3m')
    ]
    pval = float(sig_row['p_value'].iloc[0]) if len(sig_row) else 1.0
    star = '***' if pval < 0.001 else ('*' if pval < 0.05 else 'n.s.')
    ax.text(rho + 0.012, i,
            f'ρ={rho:.3f}  {star}',
            va='center', fontsize=9, color='#444441')

ax.grid(axis='x', linewidth=0.5, alpha=0.5)
sns.despine(ax=ax, left=True)

# ── Right: AUC lines across horizons ────────────────────────
ax = axes[1]
x  = np.arange(3)
ax.plot(x, auc_cf,    marker='s', linewidth=2.5, markersize=8,
        color=C_CF,    label='|ΔWARP| only (C2)')
ax.plot(x, auc_combo, marker='D', linewidth=2.5, markersize=8,
        color=C_COMBO, label='Dist + TQI + |ΔWARP| (E2)')
ax.plot(x, auc_dist,  marker='o', linewidth=2.5, markersize=8,
        color=C_DIST,  linestyle='--', label='Dist-to-limit only (A)')

for vals, col in [(auc_cf, C_CF), (auc_dist, C_DIST), (auc_combo, C_COMBO)]:
    ax.annotate(f'{vals[-1]:.3f}', (2, vals[-1]),
                textcoords='offset points', xytext=(7, 0),
                fontsize=8.5, color=col, va='center')

ax.set_xticks(x)
ax.set_xticklabels(horizons, fontsize=10)
ax.set_ylim(0.55, 0.93)
ax.set_ylabel('ROC AUC', fontsize=10)
ax.set_title('(B) Persistence prediction AUC\nacross time horizons',
             fontsize=10, pad=8)
ax.legend(fontsize=8.5, framealpha=0.7, edgecolor='#D3D1C7', loc='lower left')
ax.grid(axis='y', linewidth=0.5, alpha=0.5)
sns.despine(ax=ax)

plt.suptitle(
    'CF-derived severity metric vs traditional predictors: defect persistence prediction',
    fontsize=11, y=1.02
)
plt.tight_layout()
plt.savefig('figs/validation_A.pdf', dpi=300, bbox_inches='tight')
plt.savefig('figs/validation_A.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: figs/validation_A')


# ============================================================
# FIGURE B — Severity tier vs 3m and 6m persistence rates
# ============================================================

cf_eval_b = cf_eval.dropna(subset=['cf_warp_abs', 'persist_3m', 'persist_6m']).copy()
cf_eval_b['tier'] = cf_eval_b['cf_warp_abs'].apply(assign_tier)

tier_stats = (
    cf_eval_b.groupby('tier', sort=False)
    .agg(n=('persist_3m', 'count'),
         rate_3m=('persist_3m', 'mean'),
         rate_6m=('persist_6m', 'mean'))
    .reindex(TIER_ORDER)
    .reset_index()
)

fig, ax = plt.subplots(figsize=(7.5, 5.2))
x = np.arange(len(TIER_ORDER))
w = 0.35

bars3 = ax.bar(x - w/2, tier_stats['rate_3m'] * 100,
               width=w, color=TIER_COLORS,
               edgecolor='white', alpha=0.92,
               label='3-month persistence')
bars6 = ax.bar(x + w/2, tier_stats['rate_6m'] * 100,
               width=w, color=TIER_COLORS,
               edgecolor='white', alpha=0.55,
               hatch='//', label='6-month persistence')

# n labels rendered below the x-axis (after axis is drawn)
for xi, n_val in zip(x, tier_stats['n']):
    ax.annotate(f'n = {int(n_val)}', xy=(xi, 0), xycoords=('data', 'axes fraction'),
                xytext=(0, -38), textcoords='offset points',
                ha='center', va='top', fontsize=9, color='#6a6a6a')

for bar in list(bars3) + list(bars6):
    h = bar.get_height()
    if not np.isnan(h):
        ax.text(bar.get_x() + bar.get_width() / 2, h + 1.5,
                f'{h:.0f}%', ha='center', fontsize=8.5, color='#444441')

ax.set_xticks(x)
ax.set_xticklabels(TIER_ORDER, fontsize=9.5)
ax.set_ylim(0, 112)
ax.set_ylabel('Segments persisting (%)', fontsize=10)
# n shown in tier bars (n=626) excludes 11 samples with |ΔWARP| ≥ 0.15 in
ax.set_title(
    'Defect persistence rate by CF-derived severity tier\n'
    '(2014 reference regime; 11 samples in the top ≥0.15 in bin all persisted)',
    fontsize=10, pad=8
)
import matplotlib.patches as mpatches

# Remove the ax.legend() call and replace with this

# Two simple swatches explaining solid vs hatched only
legend_3m = mpatches.Patch(facecolor='#888780', edgecolor='white',
                            alpha=0.92, label='3-month persistence (solid)')
legend_6m = mpatches.Patch(facecolor='#888780', edgecolor='white',
                            alpha=0.55, hatch='//',
                            label='6-month persistence (hatched)')

ax.legend(handles=[legend_3m, legend_6m],
          fontsize=9, framealpha=0.7, edgecolor='#D3D1C7',
          loc='upper left')

ax.grid(axis='y', linewidth=0.5, alpha=0.4)
sns.despine(ax=ax)

plt.tight_layout()
plt.subplots_adjust(bottom=0.22)
plt.savefig('figs/validation_B.pdf', dpi=300, bbox_inches='tight')
plt.savefig('figs/validation_B.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: figs/validation_B')


# ============================================================
# FIGURE C — ROC curves (replaces calibration curve)
# ============================================================

model_display = {
    'C2_cf_warp_only':      ('|ΔWARP| only (C2)',          C_CF,    '-',  2.5),
    'E2_distance_tqi_cfwarp':('Dist + TQI + |ΔWARP| (E2)', C_COMBO, '-',  2.0),
    'A_distance_only':      ('Dist-to-limit only (A)',      C_DIST,  '--', 2.0),
    'B_tqi_only':           ('TQI only (B)',                C_TQI,   ':',  1.5),
}

fig, ax = plt.subplots(figsize=(6.5, 5.5))

for name, (label, color, ls, lw) in model_display.items():
    res = fit_results[name]
    fpr, tpr, _ = roc_curve(res['y'], res['p'])
    ax.plot(fpr, tpr,
            label=f'{label}  (AUC = {res["auc"]:.3f})',
            color=color, linestyle=ls, linewidth=lw)

# Shade under CF curve only
res_cf = fit_results['C2_cf_warp_only']
fpr_cf, tpr_cf, _ = roc_curve(res_cf['y'], res_cf['p'])
ax.fill_between(fpr_cf, tpr_cf, alpha=0.07, color=C_CF)



ax.set_xlim(-0.01, 1.01)
ax.set_ylim(-0.01, 1.01)
ax.set_xlabel('False positive rate', fontsize=11)
ax.set_ylabel('True positive rate', fontsize=11)
ax.set_title(
    'ROC curves: CF-derived severity vs baseline predictors\n'
    'Target: 1-month defect persistence  (n = 637)',
    fontsize=10, pad=8
)
ax.legend(fontsize=9, framealpha=0.8, edgecolor='#D3D1C7', loc='lower right')
ax.grid(linewidth=0.5, alpha=0.4)
sns.despine(ax=ax)

plt.tight_layout()
plt.savefig('figs/validation_C_roc.pdf', dpi=300, bbox_inches='tight')
plt.savefig('figs/validation_C_roc.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: figs/validation_C_roc')

---
## SECTION 11 — ASME Paper Verification Block
Run cells 11.1 through 11.5 **after** completing all earlier sections.  
These cells produce every number cited in the ASME submission (main.tex).


In [ ]:
# ============================================================
# 11.1 — Tier-Stratified Persistence Rates (ASME §5.3 / Tab. severity-tiers)
# Requires: cf_eval (from Section 10)
# ============================================================
import numpy as np
import pandas as pd

tier_bins  = [0.0, 0.030, 0.080, 0.150, np.inf]
tier_names = ['Marginal', 'Moderate', 'Severe', 'Critical']

cf_eval['severity_tier'] = pd.cut(
    cf_eval['cf_warp_abs'],
    bins=tier_bins, labels=tier_names, right=True
)

tier_stats = cf_eval.groupby('severity_tier', observed=False).agg(
    n          = ('cf_warp_abs',  'count'),
    persist_1m = ('persist_1m',   lambda x: 100 * x.mean()),
    persist_3m = ('persist_3m',   lambda x: 100 * x.mean()),
    persist_6m = ('persist_6m',   lambda x: 100 * x.mean()),
).round(1)

print('=== ASME Table: Tier-Stratified Persistence ===')
print(tier_stats.to_string())
print()
print('Numbers for ASME text (Section 5.3):')
for tier in tier_names:
    row = tier_stats.loc[tier]
    print(f'  {tier:10s}: n={int(row["n"]):3d}  '
          f'1m={row["persist_1m"]:.1f}%  '
          f'3m={row["persist_3m"]:.1f}%  '
          f'6m={row["persist_6m"]:.1f}%')


In [ ]:
# ============================================================
# 11.2 — Spearman ρ: CF |ΔWARP| vs Distance-to-Limit (ASME §5.4)
# Requires: cf_eval (from Section 10)
# ============================================================
from scipy.stats import spearmanr

rho, pval = spearmanr(cf_eval['cf_warp_abs'], cf_eval['distance_to_limit'])
print('=== ASME §5.4: CF severity vs physical severity ===')
print(f'  Spearman ρ = {rho:.4f}   p = {pval:.2e}   n = {len(cf_eval)}')
print()
print('  (Cited in ASME Section 5.4 and fig:cf_vs_physical caption)')


In [ ]:
# ============================================================
# 11.3 — Train Recalibrated Model  (ASME §3 / §5.9 / RQ3)
# Protocol: expanding window — add 2015 to training set.
# Recal model used for 2016 CF generation only.
# ============================================================
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score
import time

train_recal = df[df['year'].isin([2013, 2014, 2015])].copy()
X_tr_r = train_recal[FEATURES].values
y_tr_r = train_recal['defect'].values

n_neg_r = int((y_tr_r == 0).sum())
n_pos_r = int((y_tr_r == 1).sum())
SPW_r   = n_neg_r / max(1, n_pos_r)

print(f'Recal training set : {len(train_recal):,} rows')
print(f'scale_pos_weight   : {SPW_r:.2f}')

model_recal = XGBClassifier(
    n_estimators=300, max_depth=4, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    scale_pos_weight=SPW_r,
    random_state=SEED, eval_metric='logloss', verbosity=0
)
model_recal.fit(X_tr_r, y_tr_r)

# Verify recal model is not degraded
auc_2015_recal = roc_auc_score(
    test_2015['defect'], model_recal.predict_proba(test_2015[FEATURES])[:,1])
auc_2016_recal = roc_auc_score(
    test_2016['defect'], model_recal.predict_proba(test_2016[FEATURES])[:,1])
print(f'Recal AUC on 2015  : {auc_2015_recal:.4f}')
print(f'Recal AUC on 2016  : {auc_2016_recal:.4f}')
print('Recal model ready.')


In [ ]:
# ============================================================
# 11.4 — Recalibration CF Generation (ASME Table tab:recalibration)
#
# For each evaluation year, freshly generate CFs using the
# appropriate model. Success rate = "recalibrated flip rate"
# (generated CFs flip by construction; failures are infeasible
# under the model's decision geometry).
#
# 2015: base model (2013+2014 training) — same as reference
# 2016: recal model (2013+2014+2015 training) — expanding window
#
# Runtime estimate: ~5–15 min for RECAL_CAP=400 per year.
# Raise RECAL_CAP for full-cohort results.
# ============================================================

RECAL_CAP = 400   # increase if compute time permits

results_recal = {}

for year, eval_model, label in [
    (2015, model,      'base'),
    (2016, model_recal,'recal'),
]:
    df_def = df[(df['year'] == year) & (df['defect'] == 1)].copy()
    if len(df_def) > RECAL_CAP:
        df_def = df_def.sample(RECAL_CAP, random_state=SEED)

    n_total = len(df_def)
    print(f'\n{"="*55}')
    print(f'{year} defectives ({label} model)  —  n = {n_total}')
    print(f'{"="*55}')

    n_ok, n_fail = 0, 0
    t0 = time.time()
    for i, (idx, row) in enumerate(df_def.iterrows()):
        x0   = row[FEATURES].astype(float).values
        cf_x, _, _ = find_cf(eval_model, x0)
        if cf_x is not None:
            n_ok += 1
        else:
            n_fail += 1
        if (i + 1) % 100 == 0:
            elapsed = time.time() - t0
            rate    = (i + 1) / elapsed
            eta     = (n_total - i - 1) / rate
            print(f'  {i+1}/{n_total}  |  {elapsed:.0f}s elapsed  |  ~{eta:.0f}s remaining')

    success_rate = 100 * n_ok / max(1, n_total)
    results_recal[year] = {
        'model'       : label,
        'n_evaluated' : n_total,
        'n_success'   : n_ok,
        'n_failed'    : n_fail,
        'recal_flip_rate': round(success_rate, 1),
    }
    print(f'  Success: {n_ok}/{n_total}  =  {success_rate:.1f}%')
    print(f'  Failures (infeasible under model geometry): {n_fail}')

print('\n' + '='*55)
print('ASME Table tab:recalibration (recalibrated flip rates):')
print('='*55)
print(f'  Year  Stale P2   Recalibrated')
print(f'  2015  28.5%      {results_recal[2015]["recal_flip_rate"]:.1f}%  ({results_recal[2015]["model"]} model)')
print(f'  2016   6.9%      {results_recal[2016]["recal_flip_rate"]:.1f}%  ({results_recal[2016]["model"]} model)')


In [ ]:
# ============================================================
# 11.5 — ASME Numbers Verification Summary
# Prints every number cited in main.tex for cross-check.
# Run AFTER all earlier cells in this section.
# ============================================================

print('=' * 65)
print('ASME PAPER — NUMBERS VERIFICATION')
print('=' * 65)

# ── AUC values (from Section 10) ────────────────────────────
print('\n[A] PREDICTOR AUC — 2014 reference cohort (Section 5.2 / Tab. nested-models)')
key_models = ['C2_cf_warp_only', 'A_distance_only', 'B_tqi_only', 'E2_distance_tqi_cfwarp']
for outcome in ['persist_1m', 'persist_3m', 'persist_6m']:
    row = model_summary_all[
        (model_summary_all['Outcome'] == outcome) |
        (model_summary_all.get('Model', '') == 'placeholder')
    ] if 'Outcome' in model_summary_all.columns else None
    pass

# Direct extraction
try:
    for m_name, label in [
        ('C2_cf_warp_only',        '|ΔWARP| only  (C2)'),
        ('A_distance_only',        'Distance-to-limit (A)'),
        ('B_tqi_only',             'TQI only       (B)'),
        ('E2_distance_tqi_cfwarp', 'DTL+TQI+|ΔW|  (E2)'),
    ]:
        row1m = model_summary_1m[model_summary_1m['Model'] == m_name]
        a1 = float(row1m['ROC_AUC'].values[0]) if len(row1m) else float('nan')

        all3m = model_summary_all[
            (model_summary_all['Model'] == m_name) &
            (model_summary_all['Outcome'] == 'persist_3m')]
        a3 = float(all3m['ROC_AUC'].values[0]) if len(all3m) else float('nan')

        all6m = model_summary_all[
            (model_summary_all['Model'] == m_name) &
            (model_summary_all['Outcome'] == 'persist_6m')]
        a6 = float(all6m['ROC_AUC'].values[0]) if len(all6m) else float('nan')

        print(f'  {label:<30}: 1m={a1:.3f}  3m={a3:.3f}  6m={a6:.3f}')
except Exception as e:
    print(f'  AUC extraction error: {e}')

# ── Spearman correlations ─────────────────────────────────────
print('\n[B] SPEARMAN CORRELATIONS (Section 5.2 / Tab. spearman)')
try:
    for pred in ['cf_warp_abs', 'cf_dist', 'distance_to_limit', 'local_tqi']:
        sub = corr_df[corr_df['Predictor'] == pred]
        if len(sub):
            row3 = sub[sub['Outcome'] == 'persist_count_3m']
            rho3 = float(row3['Spearman_rho'].values[0]) if len(row3) else float('nan')
            pv3  = float(row3['p_value'].values[0])       if len(row3) else float('nan')
            print(f'  {pred:<25}: 3m ρ={rho3:.3f}  p={pv3:.3g}')
except Exception as e:
    print(f'  Spearman extraction error: {e}')

# ── CF vs DTL correlation ─────────────────────────────────────
try:
    from scipy.stats import spearmanr
    rho_cf_dtl, p_cf_dtl = spearmanr(cf_eval['cf_warp_abs'], cf_eval['distance_to_limit'])
    print(f'\n[C] CF |ΔWARP| vs Distance-to-limit (Section 5.4)')
    print(f'  Spearman ρ = {rho_cf_dtl:.3f}  (p={p_cf_dtl:.2e})')
except Exception as e:
    print(f'  CF-DTL correlation error: {e}')

# ── Tier persistence ─────────────────────────────────────────
print('\n[D] TIER-STRATIFIED PERSISTENCE (Section 5.3)')
try:
    for tier in tier_names:
        row = tier_stats.loc[tier]
        print(f'  {tier:10s}: n={int(row["n"]):3d}  '
              f'1m={row["persist_1m"]:.1f}%  '
              f'3m={row["persist_3m"]:.1f}%  '
              f'6m={row["persist_6m"]:.1f}%')
except Exception as e:
    print(f'  Tier stats error: {e}')

# ── Transfer flip rates ───────────────────────────────────────
print('\n[E] TRANSFER FLIP RATES (Section 5.5 / Tab. transfer)')
try:
    for label in ['Late 2014', '2015', '2016']:
        r_p1 = next((r for r in results_P1 if r['Period'] == label), None)
        r_p2 = next((r for r in results_P2 if r['Period'] == label), None)
        if r_p1 and r_p2:
            print(f'  {label:10s}: P1={r_p1["Flip Rate (%)"]:.1f}%  '
                  f'P2={r_p2["Flip Rate (%)"]:.1f}%  '
                  f'N={r_p1["Defects"]}')
except Exception as e:
    print(f'  Transfer flip rates error: {e}')

# ── Recalibration ─────────────────────────────────────────────
print('\n[F] RECALIBRATION FLIP RATES (Section 5.9 / Tab. recalibration)')
try:
    for year in [2015, 2016]:
        r = results_recal[year]
        print(f'  {year}: stale P2 = {"28.5" if year==2015 else "6.9"}%  →  '
              f'recalibrated = {r["recal_flip_rate"]:.1f}%  '
              f'({r["model"]} model, n={r["n_evaluated"]})')
except Exception as e:
    print(f'  Recalibration results error: {e} — run cell 11.4 first')

# ── KS statistics ─────────────────────────────────────────────
print('\n[G] KS STATISTICS — XLEVEL (Section 5.5)')
print('  (From Section 4 / cell 21 — verify against: 0.083, 0.161, 0.430)')

# ── CF generation stats ───────────────────────────────────────
print('\n[H] CF GENERATION STATS — 2014 (Section 5.1 / Tab. cf-summary)')
try:
    n_total_2014 = len(df[(df['year']==2014) & (df['defect']==1)])
    n_cf = len(cf_df)
    print(f'  2014 defectives : {n_total_2014}')
    print(f'  CFs generated  : {n_cf}  ({100*n_cf/n_total_2014:.1f}%)')
    print(f'  Failures        : {n_total_2014 - n_cf}')
    print(f'  Mean |ΔWARP|    : {cf_df["d_WARP_62"].abs().mean():.4f} in')
    print(f'  Mean CF dist    : {cf_df["cf_dist"].mean():.4f}')
except Exception as e:
    print(f'  CF stats error: {e}')

print('\n' + '=' * 65)
print('END OF ASME VERIFICATION')
print('=' * 65)
